# Origins-of-life vent campaign

_Investigation `origins-of-life-vent` — coder reproduction notebook._

**Question.** Could the first cells have originated at alkaline hydrothermal vents? Can a physically plausible early-Earth vent–ocean interface generate and sustain, in order, the free energy, reduced carbon, self-amplifying chemistry, concentration, and compartments a protocell needs — without inserting metabolism, membranes, or heredity by assumption?

A staged, falsifiable Vivarium campaign testing the alkaline-vent theory for the origin of cells — treating a cellular origin as coupled transitions from geochemistry to bioenergetic autonomy, grounded in serpentinization-driven alkaline hydrothermal vents, with explicit competing scenarios.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Classical alkaline-vent (Russell/Martin/Lane/Sojo)**
- **2026 unfavorable reassessment (Tutolo; Jackson)**
- **Shallow-vent (Barge & Price)**
- **Terrestrial geothermal (Mulkidjanian)**
- **Cyanosulfidic/photochemical (Sutherland)**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-genesis/viva-genesis').is_dir():
    REPO = Path('/home/runner/work/viva-genesis/viva-genesis')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_genesis.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: The Serpentinizing Vent (`og-00-vent-reactor-baseline`)

**Question.** Can a Lost City–like, serpentinization-fed vent–ocean interface reproduce observed fluid mixing, heat transport, mineral precipitation and H2/alkalinity delivery under field-constrained conditions — with no origin-of-life claim?

**Objective.** Build a reduced vent–ocean reactor and validate it against Lost City field ranges to establish a geochemical/spatial testbed.

**Hypothesis.** A 1D/2D porous-wall reactor with prescribed boundary fluxes reproduces published Lost City ranges (T, pH, H2, DIC) and qualitative precipitation trends while conserving mass and charge.

**Claim.** Habitability is a bounded REGION of the operating space, not a single tuned run: across the (olivine reaction rate × flow velocity) grid the real PHREEQC + FiPy serpentinizing vent settles inside the Lost City window — T 40–91 °C AND pH 9.0–10.8 AND a positive H2 flux — over ~80% of the swept grid, and the baseline operating point sits safely inside it (T 50.6 °C, pH 9.37) with sustained H2, exact charge balance (|charge|=0) and FiPy flux divergence within the contract tolerance (4.9e-5 vs the 1e-3 bound).


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: serpentinization as boundary flux
- B: flow/mixing/precipitation, evolving porosity

**Proposed models / methods**

- PHREEQC or Reaktoro (speciation + mineral kinetics)
- FiPy finite-volume advection–diffusion–reaction
- prescribed vent source term

**Key assumptions**

- Hadean ocean chemistry parameterized separately — never reuse modern seawater as the ancient prior.
- Partial equilibrium: a prescribed olivine dissolution rate (flow/T-driven) feeds PHREEQC equilibrium speciation each step — fast aqueous/mineral equilibrium vs a rate-limited rock reaction, standard for reactive-transport.
- The heat-partition fraction (fraction of serpentinization enthalpy retained in the fluid parcel) is CALIBRATED to the observed 40-91 C vent temperature; the chemistry (H2, pH, pe, charge) is emergent from PHREEQC, not tuned.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Vent reactor baseline` | `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor`**

A reusable alkaline-vent reactor module. Module A = genuine PHREEQC serpentinization (phreeqpython + vendored llnl.dat): olivine dissolution + serpentine/brucite/magnetite equilibrium gives dissolved H2 (reducing power), alkalinity, heat, pe, and a charge-balanced fluid. Module B = genuine FiPy finite-volume advection-diffusion-reaction column transporting those sources across the vent-ocean interface and setting outlet T/pH/porosity. PHREEQC conserves mass+charge and FiPy's FVM conserves flux, satisfying the og-00 contract invariants; the coupled model is driven by a real IPhreeqc speciation (needs phreeqpython) and a real FiPy FVM solve (needs fipy).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = {
    'flow_velocity': 0.001,   # Boundary water flux through the reaction zone (m/s)
    'olivine_rate' : 0.15,   # Olivine dissolution rate (mol per flow-saturated reaction step)
    'nx'           : 40,   # FiPy finite-volume cells across the interface column
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `H2` | float | molar | >= 0.0 |
| `DIC` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `H2_flux` | float | mole/second | >= 0.0 |
| `alkalinity_flux` | float | mole/second |  |
| `heat_flux` | float | watt |  |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-00-vent-reactor-baseline ===
STUDY = 'og-00-vent-reactor-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Vent_reactor_baseline = 100
INTERVAL_Vent_reactor_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = materialize_composite('viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor['state']}, core=core)
            _comp.run(STEPS_Vent_reactor_baseline)
        print('ran', 'Vent reactor baseline', f'(viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor) for', STEPS_Vent_reactor_baseline, 'steps')
    except Exception as _e:
        print(f"Vent reactor baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-cross-section**


In [ ]:
# vent-cross-section
show_viz(_render_one('local:VentCrossSection', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 60, 'title': 'Real 2-D vent cross-section (FiPy)', 'max_frames': 20}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

**vent-reactor-timeseries**


In [ ]:
# vent-reactor-timeseries
show_viz(_render_one('local:VentReactorTimeseries', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 120, 'title': 'Real vent reactor (PHREEQC + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

**vent-serpentinization-engine**


In [ ]:
# vent-serpentinization-engine
show_viz(_render_one('local:VentSerpentinizationEngine', {'n_steps': 120, 'nx': 40, 'title': 'The serpentinization engine (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**vent-habitability-map**


In [ ]:
# vent-habitability-map
show_viz(_render_one('local:VentHabitabilityMap', {'grid_json': 'workspace/studies/og-00-vent-reactor-baseline/artifacts/vent_habitability_grid.json', 'title': 'The Lost City habitability window (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vent-temperature-in-lostcity-range _(primary)_ | kind=derived path=settled_temperature_c detail=last-quartile mean outlet temperature, degrees C | op range low 40.0 high 91.0 detail settled T in [40, 91] C |
| h2-flux-sustained _(primary)_ | kind=derived path=min_sustained_H2_flux detail=minimum H2 source flux over the back half of the run | op > value 1e-09 detail min sustained H2 flux > 0 |
| charge-conserved _(primary)_ | kind=derived path=max_abs_charge_imbalance detail=worst-case |charge imbalance| over the run (eq/kgw) | op <= value 1e-06 detail |charge imbalance| < 1e-6 throughout |
| spatial-flux-conserved _(primary)_ | kind=derived path=max_flux_divergence_residual detail=worst-case normalized FVM flux-divergence residual over the run | op <= value 0.001 detail max |flux divergence residual| < 1e-3 (direct LU solve ~ machine eps) |
| vent-pH-in-lostcity-range _(primary)_ | kind=derived path=settled_pH detail=last-quartile mean outlet pH | op range low 9.0 high 10.8 detail settled pH in [9.0, 10.8] |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Lost City ~40-91 C, pH ~9-10.8, >=30 ky activity | outlet T,pH; run duration | Kelley 2001 / Fruh-Green 2003 / Ludwig 2006 | Met by the real coupled model: settled T ~51 C and pH ~9.4 are both in the Lost City window, H2 reducing power is sustained, and mass/charge/flux conservation hold. pH was closed by reacting peridotite (olivine + pyroxene) so the Ca-OH endmember buffers the fluid to field pH — a real PHREEQC result. |


## Study: The First Carbon (`og-01-geology-to-reduced-carbon`)

**Question.** Which combinations of T, pH, pressure, H2, CO2 and mineralogy generate sustained net organic-carbon flux across the interface — reproducing Hudson/Preiner/Altair regimes before any extrapolation?

**Objective.** Add H2-driven CO2 reduction, reproduce the three benchmark experiments, and map the parameter region of sustained net organic-carbon flux.

**Hypothesis.** Over Fe/Ni/S surfaces and across the pH gradient, H2-driven CO2 reduction yields formate/acetate/pyruvate at lab-calibrated rates within a bounded (T,pH,H2,pressure) region; the Tutolo-2026 in-situ arm roughly halves the flux, and the feature-ablation controls (no gradient / no H2 / no catalyst) zero it.

**Claim.** H2-driven CO2 reduction over Fe/Ni/S reproduces the Preiner/Hudson lab yields and net organic-carbon flux in a bounded favorable regime; the Tutolo-2026 arm shows the usable proton-motive gradient collapses at in-situ T/P (427 -> 43 mV), roughly halving the sustained flux; only the no-gradient / no-H2 / no-catalyst controls zero it.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- C: pH-gradient electrochemistry
- D: mineral-catalyzed carbon fixation

**Proposed models / methods**

- Nernst–Planck–Poisson (electroneutral) on FiPy/FEniCSx
- Cantera/custom heterogeneous kinetics
- eQuilibrator feasibility
- PyMC Bayesian calibration to Preiner/Hudson/Altair

**Key assumptions**

- Encode pH(T,P,ionic_strength), never laboratory pH as a constant environmental state.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Geology to reduced carbon — favorable` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | — |
| `Tutolo-2026 in-situ (unfavorable arm)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.1 |

Declared parameter sets (`study.yaml` variants):

- **tutolo-2026** — `gradient_efficiency=0.1`
- **control-no-gradient** — `gradient_efficiency=1e-06`
- **control-no-h2** — `h2=1e-12`
- **control-no-catalyst** — `catalyst_active=1e-06`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy finite-volume Nernst-Planck pH-gradient electrochemistry (pmf, current density; needs fipy). Module D = genuine eQuilibrator ΔG' feasibility gate (needs equilibrator_api) + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |
| `ionic_strength` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `current_density` | float | ampere/meter**2 |  |
| `electric_potential` | float | volt |  |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-geology-to-reduced-carbon ===
STUDY = 'og-01-geology-to-reduced-carbon'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Geology_to_reduced_carbon_favorable = 100
INTERVAL_Geology_to_reduced_carbon_favorable = 1.0
STEPS_Tutolo_2026_in_situ_unfavorable_arm = 100
INTERVAL_Tutolo_2026_in_situ_unfavorable_arm = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Geology_to_reduced_carbon_favorable)
        print('ran', 'Geology to reduced carbon — favorable', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Geology_to_reduced_carbon_favorable, 'steps')
    except Exception as _e:
        print(f"Geology to reduced carbon — favorable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Tutolo_2026_in_situ_unfavorable_arm)
        print('ran', 'Tutolo-2026 in-situ (unfavorable arm)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Tutolo_2026_in_situ_unfavorable_arm, 'steps')
    except Exception as _e:
        print(f"Tutolo-2026 in-situ (unfavorable arm) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**proton-gradient-profile**


In [ ]:
# proton-gradient-profile
show_viz(_render_one('local:ProtonGradientProfile', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 80, 'title': 'Proton gradient → reduced carbon'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-timeseries**


In [ ]:
# carbon-fixation-timeseries
show_viz(_render_one('local:CarbonFixationViz', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 120, 'title': 'Geology to reduced carbon (eQuilibrator + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-phase-map**


In [ ]:
# carbon-fixation-phase-map
show_viz(_render_one('local:CarbonFixationPhaseMap', {'title': 'The carbon-fixation landscape (Decision view)', 'grid_json': 'workspace/studies/og-01-geology-to-reduced-carbon/artifacts/carbon_fixation_phase_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| formate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_formate_mM detail=batch formate yield mM=None | op range low 50.0 high 220.0 detail formate in Preiner range (~200 mM ceiling) |
| acetate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_acetate_uM detail=batch acetate yield uM=None | op range low 20.0 high 150.0 detail acetate in Preiner range |
| pyruvate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_pyruvate_uM detail=batch pyruvate yield uM=None | op range low 2.0 high 15.0 detail pyruvate in Preiner range |
| net-carbon-flux-nonzero-region _(primary)_ | kind=derived path=peak_net_carbon_flux detail=peak organic-C fixation rate | op > value 1e-06 detail peak net carbon flux > 0 (viable region) |
| free-energy-directionality-logged _(primary)_ | kind=derived path=free_energy_logged_fraction detail=fraction of steps with logged ΔG' | op >= value 1.0 detail ΔG' logged every step |
| chemiosmotic-backpressure-loads-gradient _(primary)_ | kind=derived path=pmf_backpressure_sag_frac detail=peak fractional pmf sag while fixation draws the proton reservoir down | op range low 0.05 high 0.95 detail a meaningful back-pressure sag (gradient is spent) without total collapse |
| nernst-planck-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_electro_flux_residual detail=max |FVM residual| | op <= value 0.01 detail max |flux divergence residual| small |
| ph-gradient-drives-formate _(primary)_ | kind=cross_run detail=favorable formate vs no-gradient / no-H2 / no-catalyst | op controls_absent detail formate present with gradient <1 mM in every control None |
| in-situ-ph-correction _(diagnostic)_ | kind=cross_run detail=favorable pmf vs Tutolo-2026 pmf | op arms_reported detail report both arms; flag if the Tutolo pmf collapses |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Formate up to ~200 mM, acetate ~100 uM, pyruvate ~10 uM at 100 C | product concentrations by mineral/T/P | Preiner 2020 | Met by the favorable arm — batch formate ~200 mM, acetate ~100 uM, pyruvate ~10 uM (Preiner hierarchy), via real eQuilibrator-gated, lab-calibrated mineral kinetics. |
| CO2+H2->formate across pH gradient at room T, ~1.5 bar | formate yield vs delta-pH + controls | Hudson 2020 | Met — formate is present with the pH-gradient proton-motive force and collapses to <1 mM in every control (no gradient / no H2 / no catalyst). |


## Study: The First Metabolism (`og-02-reduced-carbon-to-protometabolism`)

**Question.** Under continuous feed and washout, do reaction subnetworks become self-reinforcing — persistent cycles, flux amplification, product reuse, autocatalytic motifs — without imposed product accumulation?

**Objective.** Insert an experimentally-motivated reaction graph and test whether metabolism-like subnetworks emerge under flow.

**Hypothesis.** A curated graph (formate/acetate/pyruvate/glyoxylate + rTCA fragments) seeded with Muchowska/Moran edges sustains nonzero throughput and at least one autocatalytic motif within a feasible resource budget.

**Claim.** A curated rTCA-fragment network under continuous feed + washout sustains nonzero carbon throughput, exhibits an autocatalytic motif (OAA self-amplification, autocatalytic gain peaking ~29.5), and forms a single connected reaction network — metabolism-like behavior emerging without imposed product accumulation.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- E: protometabolic network formation
- D: surface chemistry

**Proposed models / methods**

- SBML + libRoadRunner/Tellurium or COPASI
- SciPy solve_ivp
- element-balance + thermodynamic-consistency checks before kinetic fitting
- GillesPy2 for low-copy stochastic regimes

**Key assumptions**

- Rate laws/mineral dependencies carry provenance; autocatalysis is an emergent observable, not an assumption.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Reduced carbon to protometabolism` | `viva_genesis.composites.protometabolism.protometabolic_network` | 0 | — |

Declared parameter sets (`study.yaml` variants):

- **control-no-co2** — `feed_CO2=1e-12`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.protometabolism.protometabolic_network`**

A reusable protometabolic reaction-network module. A genuine libRoadRunner/Tellurium SBML ODE integration (needs tellurium) of an rTCA-fragment CSTR (pyruvate carboxylation + lumped autocatalytic rTCA turn + reductive branch) fed the og-01 reduced-carbon products under continuous feed+washout, with a networkx connectivity check on the species-reaction graph (needs networkx). Emergent: sustained carbon throughput, an autocatalytic motif (gain>1), and a connected reaction network.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_protometabolism_protometabolic_network = {
    'feed_pyruvate': 0.0001,   # Pyruvate feed set-point (M) from og-01 reduced carbon
    'feed_CO2'     : 0.002,   # CO2 feed set-point (M); ~0 = no-carbon control
    'k_rtca'       : 800.0,   # Autocatalytic rTCA-turn rate constant (Fe-promoted)
    'k_wash'       : 0.05,   # CSTR washout / dilution rate (1/s); the feed-vs-washout axis of the gain phase map
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-02-reduced-carbon-to-protometabolism ===
STUDY = 'og-02-reduced-carbon-to-protometabolism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Reduced_carbon_to_protometabolism = 100
INTERVAL_Reduced_carbon_to_protometabolism = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_protometabolism_protometabolic_network = materialize_composite('viva_genesis.composites.protometabolism.protometabolic_network', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_protometabolism_protometabolic_network['state']}, core=core)
            _comp.run(STEPS_Reduced_carbon_to_protometabolism)
        print('ran', 'Reduced carbon to protometabolism', f'(viva_genesis.composites.protometabolism.protometabolic_network) for', STEPS_Reduced_carbon_to_protometabolism, 'steps')
    except Exception as _e:
        print(f"Reduced carbon to protometabolism not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protometabolic-network**


In [ ]:
# protometabolic-network
show_viz(_render_one('local:ProtometabolicNetwork', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 60, 'title': 'The protometabolic network igniting'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None)))

**protometabolism-network**


In [ ]:
# protometabolism-network
show_viz(_render_one('local:ProtometabolismViz', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 120, 'title': 'Protometabolic network (libRoadRunner)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None)))

**protometabolism-gain-phase-map**


In [ ]:
# protometabolism-gain-phase-map
show_viz(_render_one('local:ProtometabolismGainPhaseMap', {'title': 'The protometabolic ignition landscape (Decision view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sustained-carbon-throughput _(primary)_ | kind=derived path=settled_net_carbon_flux detail=steady net carbon fixation rate | op > value 1e-06 detail net throughput > 0 (no imposed accumulation) |
| autocatalytic-gain-present _(primary)_ | kind=derived path=peak_autocatalytic_gain detail=peak OAA self-amplification ratio | op >= value 1.0 detail >= 1 reinforcing/autocatalytic motif |
| network-closure-vs-oneoff _(supporting)_ | kind=derived path=network_connected_final detail=networkx connectivity of the species-reaction graph | op >= value 1.0 detail connected network (not one-offs) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Fe2+ promotes 9/11 TCA intermediates + 5 universal precursors from pyruvate/glyoxylate | edge yields + network reachability | Muchowska 2019 | Represented as the lumped reductive-TCA turn (the curated rTCA class on libRoadRunner), graded; edge-by-edge 9/11 reachability is the Phase-3 reaction_universe comparison, not yet run |


## Study: The First Individual (`og-03-dilution-to-localized-individuals`)

**Question.** Where in pore geometry (openness × length) does a real thermophoretic + convective trap concentrate a dilute vent feed across the localization threshold — the point at which a bounded, concentrated "individual" forms rather than washing through?

**Objective.** Add pore geometry, adsorption, convection and thermophoresis; find where retention + concentration cross persistence thresholds.

**Hypothesis.** Thermophoresis + convection + adsorption in evolving pores raise local concentration enough that synthesis outruns hydrolysis/leakage in a bounded geometry region; elsewhere washout dominates.

**Claim.** Localization is real and robust: a real FiPy thermophoretic + convective trap concentrates a dilute vent feed into a bounded REGION of pore geometry, mapped natively over (porosity × pore length) — long, tight pores localize strongly (a persistent local "individual" emerges) while open / short pores wash the feed through (enrichment ~1). The EXTENT of enrichment the favourable trap reaches (reduced-model estimate ~1e6–1e7-fold at accumulation Péclet ≈ 18) is a solver-sensitive extrapolation above Pe ≈ 13, not a precise resolved result — the defensible claim is that localization occurs, not the exact fold-magnitude (Baaske 2007).


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- F: transport/concentration/retention
- B: evolving pores

**Proposed models / methods**

- FiPy finite-volume with Soret term
- OpenPNM/PoreSpy pore geometry
- reproduce Baaske enrichment
- Damkohler/Peclet/Soret scaling

**Key assumptions**

- Concentration arises from the balance of advection, diffusion, thermodiffusion, adsorption, precipitation and degradation — not assumed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Dilution to localized individuals` | `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration` | 0 | flow_velocity=0.0001, porosity=0.3, pore_length=0.005, delta_T=30.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration`**

A reusable thermophoretic-concentration module for the Baaske thermal trap. Module F = genuine FiPy finite-volume accumulation column (needs fipy): thermophoresis (Soret) + buoyant convection concentrate a dilute feed, enrichment exponential in pore length (>10^8 favorable, ~1 in controls). flow_velocity=0 / porosity=1 give the no-trap controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = {
    'flow_velocity': 0.0001,   # Buoyant convective flow in the pore (m/s); 0 = no-convection control
    'porosity'     : 0.3,   # Pore porosity/openness (0-1); ~1 = open-pore (wash-through) control
    'temperature'  : 333.15,   # Mean pore temperature (K)
    'pore_length'  : 0.005,   # Pore length (m); longer = more enrichment
    'delta_T'      : 30.0,   # Cross-pore temperature difference (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `enrichment_factor` | float | dimensionless | >= 1.0 |
| `residence_time` | float | second | >= 0.0 |

**Conditions**

- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-03-dilution-to-localized-individuals ===
STUDY = 'og-03-dilution-to-localized-individuals'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Dilution_to_localized_individuals = 100
INTERVAL_Dilution_to_localized_individuals = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = materialize_composite('viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration['state']}, core=core)
            _comp.run(STEPS_Dilution_to_localized_individuals)
        print('ran', 'Dilution to localized individuals', f'(viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration) for', STEPS_Dilution_to_localized_individuals, 'steps')
    except Exception as _e:
        print(f"Dilution to localized individuals not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**thermophoretic-trap**


In [ ]:
# thermophoretic-trap
show_viz(_render_one('local:ThermophoreticTrapViz', {'composite': 'viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', 'title': 'The thermal trap concentrates an individual'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None)))

**pore-ignition-coupling**


In [ ]:
# pore-ignition-coupling
show_viz(_render_one('local:PoreIgnitionCoupling', {'title': 'Localization lifts the feed across the ignition threshold (Mechanism view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

**pore-localization-map**


In [ ]:
# pore-localization-map
show_viz(_render_one('local:PoreLocalizationMap', {'grid_json': 'workspace/studies/og-03-dilution-to-localized-individuals/artifacts/pore_localization_grid.json', 'title': 'The pore-localization frontier (Decision view)'}, RUNS_DB, STUDY_YAML))

**multisolute-colocalization-map**


In [ ]:
# multisolute-colocalization-map
show_viz(_render_one('local:PoreColocalizationMap', {'grid_json': 'workspace/studies/og-03-dilution-to-localized-individuals/artifacts/colocalization_grid.json', 'title': 'Does the prebiotic mixture co-localize? (Phase 4 Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| thermophoretic-enrichment-reproduces-baaske _(primary)_ | kind=derived path=settled_accumulation_peclet detail=ln(enrichment) = beta × pore length (reduced column) | op arms_reported detail reduced-column Pe_acc ≈ 18.4 (~10^8-fold) reported as the closed-cell/reduced-model figure; the resolved FEM does not confirm it past the convergence-verified Pe_acc ≈ 13 |
| enrichment-resolved-within-convergence _(primary)_ | kind=derived path=log10_enrichment detail=localization occurs within the resolvable range (positive enrichment agreement region)=None | op > value 0.0 detail log10_enrichment > 0 (the resolved trap concentrates above inflow within Pe_acc ≲ 13) |
| retention-beats-destruction _(primary)_ | kind=derived path=log10_enrichment detail=log10 of the settled enrichment fold (reduced column; resolved value within Pe_acc ≲ 13) | op arms_reported detail reduced-column ≥10^6-fold reported as the closed-cell figure; net accumulation within demonstrated convergence is the gated claim (enrichment-resolved-within-convergence) |
| open-pore-control-does-not-concentrate _(primary)_ | kind=cross_run detail=favorable enrichment vs open-pore control | op controls_absent detail enrichment collapses to ~1 in the open-pore control |
| enrichment-exponential-in-pore-length _(diagnostic)_ | kind=cross_run detail=favorable vs 10× shorter pore | op arms_reported detail short pore enriches ≥100× less than the favorable pore |
| spatial-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_trap_flux_residual detail=max |FVM flux-divergence residual| | op <= value 0.01 detail FVM residual small (direct LU solve) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| >10^8-fold nucleotide accumulation in favorable modeled pore geometries | enrichment vs geometry | Baaske 2007 | Partially met by the real FiPy thermophoretic trap — the favourable pore reaches the extreme-accumulation regime (accumulation Péclet 18.4) and the native (porosity × pore length) phase map localizes at ~17% of the grid with open/short-pore controls not concentrating. The exact Baaske fold-magnitude (reduced-model ~1e6–1e7) is solver-sensitive above Pe ≈ 13 (FiPy↔FEniCS diverge by up to ~5.6 decades there), so localization is confirmed but the precise fold is not. |


## Study: The First Boundary (`og-04-mineral-compartments-to-protocells`)

**Question.** Do mixed-amphiphile organic boundaries enhance persistence without cutting the system off from geochemical energy?

**Objective.** Introduce mixed amphiphiles + membrane assembly; measure whether organic boundaries enhance persistence without isolating the system.

**Hypothesis.** Over a salt/pH/divalent-cation/T window, vesicles form and selectively retain while remaining permeable enough to exploit the external gradient — a non-empty individuality-vs-access region exists.

**Claim.** Mixed single-chain amphiphiles form viable vesicles across the alkaline / seawater-salt window where single fatty acids fail (Jordan 2019), and those compartments retain internal chemistry while staying permeable to the gradient — a non-empty individuality-without-isolation region.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- G: amphiphiles, mineral pores and protocells

**Proposed models / methods**

- custom coarse-grained membrane kinetics
- Smoldyn/ReaDDy spatial stochastic
- OpenMM/LAMMPS to calibrate permeability laws

**Key assumptions**

- Mineral pores provide initial individuality; organic boundaries are a later, condition-dependent transition.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Mineral compartments to protocells` | `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell` | 0 | mix_fraction=0.4, pH=9.0, ionic_strength=0.35, temperature=318.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell`**

A reusable mixed-amphiphile protocell module (Module G). Coarse-grained vesicle self-assembly + permeability calibrated to Jordan 2019: mixed single-chain amphiphiles broaden the pH / salt / divalent stability window and form vesicles that retain chemistry while staying permeable to the gradient (individuality without isolation). mix_fraction≈0 / high salt / acidic pH give the controls. Coarse-grained reproduction (not wrapped MD).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = {
    'mix_fraction'  : 0.4,   # Co-surfactant fraction (0 = single fatty acid; higher = broader Jordan window)
    'pH'            : 9.0,   # Environment pH (alkaline vent)
    'ionic_strength': 0.35,   # Ionic strength (M); ~seawater 0.35
    'temperature'   : 318.15,   # Temperature (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `ionic_strength` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `membrane_area` | float | meter**2 | >= 0.0 |
| `permeability` | float | meter/second | >= 0.0 |
| `compartment_volume` | float | liter | >= 0.0 |
| `viable_compartments` | integer | dimensionless | >= 0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-04-mineral-compartments-to-protocells ===
STUDY = 'og-04-mineral-compartments-to-protocells'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Mineral_compartments_to_protocells = 100
INTERVAL_Mineral_compartments_to_protocells = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = materialize_composite('viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell['state']}, core=core)
            _comp.run(STEPS_Mineral_compartments_to_protocells)
        print('ran', 'Mineral compartments to protocells', f'(viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell) for', STEPS_Mineral_compartments_to_protocells, 'steps')
    except Exception as _e:
        print(f"Mineral compartments to protocells not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protocell-boundary-world**


In [ ]:
# protocell-boundary-world
show_viz(_render_one('local:ProtocellBoundaryWorld', {'mix_fraction': 0.4, 'ionic_strength': 0.35, 'pH': 9.0, 'title': 'The compartment and the gradient (World view)'}, RUNS_DB, STUDY_YAML))

**protocell-window**


In [ ]:
# protocell-window
show_viz(_render_one('local:ProtocellWindowViz', {'composite': 'viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', 'title': 'The protocell window broadens — mixed amphiphiles (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None)))

**protocell-retention-access**


In [ ]:
# protocell-retention-access
show_viz(_render_one('local:ProtocellPhaseMap', {'grid_json': 'workspace/studies/og-04-mineral-compartments-to-protocells/artifacts/protocell_grid.json', 'title': 'A boundary that retains without isolating (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vesicles-form-in-window _(primary)_ | kind=derived path=settled_viable_compartments detail=viable vesicle count in the favorable arm | op > value 0.0 detail vesicles form (>0) in the mixed-amphiphile window |
| retention-without-isolation _(primary)_ | kind=derived path=settled_individuality_access detail=retention × access | op >= value 0.3 detail both retention and access nonzero (a non-empty individuality-vs-access region) |
| mixed-amphiphile-advantage-over-single _(primary)_ | kind=cross_run detail=favorable (mixed) viable vs single-fatty-acid control | op controls_absent detail mixed forms vesicles; single fatty acid does not at the same salt |
| membrane-transition-robust _(diagnostic)_ | kind=cross_run detail=favorable vs extreme-salt and acidic controls | op arms_reported detail extreme-salt and acidic controls abolish vesicles (survival region is bounded) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mixed single-chain amphiphiles form vesicles across broader alkaline pH/salinity/divalent/T than single-fatty-acid models | vesicle stability map | Jordan 2019 | Met by the coarse-grained Jordan-calibrated protocell — mixed amphiphiles form ~8000 viable vesicles at alkaline vent pH and seawater salt where a single fatty acid forms none, and the (co-surfactant × ionic strength) feasibility map marks a viable region (individuality > 0.1) at 63/256 points. |


## Study: Owning Its Energy (`og-05-environmental-to-internal-bioenergetics`)

**Question.** Can selection move from externally imposed gradients toward internally maintained electrochemical states (catalytic ion transport then pumping)?

**Objective.** Allow catalytic ion transport and active pumping; test whether selection favors internalized gradient generation.

**Hypothesis.** When membrane permeability, Na+/H+ coupling and catalyst efficiency vary under selection, systems with increasing internal gradient generation win on throughput + persistence — but only if membranes stay sufficiently leaky (Sojo constraint).

**Claim.** Survival off a collapsing gradient is a bounded REGION of (pump catalyst efficiency × gradient floor), mapped by a real replicator sweep — a lineage persists either by riding a floor that never falls too far OR by internalizing gradient generation, but internalizing only pays above the Sojo cost line (η ≈ 0.30). At the operating point an efficient pump internalizes (pump fraction 0→0.95) and survives while a weak pump collapses — the Sojo-2014 leaky-membrane constraint on bioenergetic internalization.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H: bioenergetic internalization

**Proposed models / methods**

- evolutionary control loop over membrane parameters
- reward = throughput + persistence
- Sojo leaky-membrane model as baseline

**Key assumptions**

- Internalization is an evolutionary control problem, not a scripted upgrade.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Environmental to internal bioenergetics` | `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization` | 0 | catalyst_efficiency=1.0, v_start=0.4, decline_rate=0.01 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization`**

A reusable bioenergetic-internalization module (Module H). An evolutionary control loop (replicator dynamics, Sojo-2014 leaky-membrane fitness) under a declining external gradient: as the geochemical pmf collapses, an efficient-pump population internalizes gradient generation and persists, while a weak-pump control collapses (the Sojo constraint). catalyst_efficiency sets the arm. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = {
    'catalyst_efficiency': 1.0,   # Active-pump (Na+/H+) efficiency (1.0 efficient; ~0.3 weak-pump control)
    'v_start'            : 0.4,   # Initial external gradient (V)
    'decline_rate'       : 0.01,   # External-gradient decline per step (V/s)
    'gradient_floor'     : 0.02,   # Floor the external gradient decays to (V); low floor forces internalization-or-die
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `electric_potential` | float | volt |  |
| `catalyst_efficiency` | float | dimensionless | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |
| `internalization` | float | dimensionless | [0.0, 1.0] |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `mean_permeability` | float | meter/second | >= 0.0 |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-05-environmental-to-internal-bioenergetics ===
STUDY = 'og-05-environmental-to-internal-bioenergetics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Environmental_to_internal_bioenergetics = 100
INTERVAL_Environmental_to_internal_bioenergetics = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = materialize_composite('viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization['state']}, core=core)
            _comp.run(STEPS_Environmental_to_internal_bioenergetics)
        print('ran', 'Environmental to internal bioenergetics', f'(viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization) for', STEPS_Environmental_to_internal_bioenergetics, 'steps')
    except Exception as _e:
        print(f"Environmental to internal bioenergetics not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**bioenergetic-internalization**


In [ ]:
# bioenergetic-internalization
show_viz(_render_one('local:BioenergeticInternalizationViz', {'composite': 'viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', 'title': 'Selection internalizes the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None)))

**bioenergetic-provenance**


In [ ]:
# bioenergetic-provenance
show_viz(_render_one('local:BioenergeticProvenanceViz', {'title': 'Energy provenance — external to internal (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**bioenergetic-survival-map**


In [ ]:
# bioenergetic-survival-map
show_viz(_render_one('local:BioenergeticSurvivalMap', {'grid_json': 'workspace/studies/og-05-environmental-to-internal-bioenergetics/artifacts/bioenergetic_survival_grid.json', 'title': 'Survival off the collapsing gradient (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| selection-internalizes-gradient-generation _(primary)_ | kind=derived path=final_internalization detail=active-pump fraction of the surviving population at the collapsed gradient | op >= value 0.6 detail the population internalizes (pump fraction >= 0.6) as the external gradient falls |
| reduced-gradient-dependence _(primary)_ | kind=derived path=final_survivors detail=surviving strategy mass at the collapsed external gradient | op >= value 0.5 detail internalizing lineage persists (survivors >= 0.5) at the collapsed gradient |
| sojo-constraint-weak-pump-collapses _(primary)_ | kind=cross_run detail=efficient-pump survivors vs weak-pump control | op controls_absent detail the weak-pump control collapses (survivors ~0) where the efficient pump persists |
| energy-cost-accounted _(supporting)_ | kind=derived path=final_internalization detail=internalized population has net-positive flux | op > value 0.0 detail a net-positive-flux internalized region exists |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA-like cells can use proton gradients only if membranes remain sufficiently leaky | permeability vs viability | Sojo 2014 / Lane 2012 | Met by the real replicator selection loop — the survival Decision map over (pump efficiency × gradient floor) is bounded by the Sojo cost line (η ≈ 0.30): efficient pumps internalize and persist (~94% of the grid survives) while weak pumps collapse at the gradient transition. |


## Study: The First Heredity (`og-06-chemistry-to-heredity`)

**Question.** Does inherited state improve persistence/throughput — comparing RNA-like, peptide-assisted, compositional and hybrid heredity — rather than template replication in isolation? (Do NOT presume RNA.)

**Objective.** Introduce competing heredity modules and test whether inherited state improves persistence under feasible budgets.

**Hypothesis.** At least one heredity module raises lineage persistence + information retention above its non-heritable control under a feasible resource budget; the winner is not assumed a priori.

**Claim.** An inherited chemical state improves lineage fitness above its replication cost when it clears the Eigen error threshold — but heredity need NOT presume a long high-fidelity RNA template. An agent-based lineage shows template heredity suffers the length catastrophe (Q=fidelity^length collapses with length) while compositional (GARD) and template-focused hybrid heredity persist length-independently (functional persistence ~0.88, seed-robust across the 5 graded seeds — 0.878±0.005, and 8/8 in a wider sweep) above the non-heritable control, which loses the parent↔offspring correlation and collapses. (A mean-field quasispecies arm still ranks a long high-fidelity template above a cheaper peptide module above non-inheritance when all clear the threshold.)


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- J: polymers, information and heredity (as competing modules)

**Proposed models / methods**

- GillesPy2/StochPy SSA
- rule-based chemistry (BioNetGen/NFsim) for sequence combinatorics
- separate catalyst discovery from inheritance
- explicit fidelity/cost/competition

**Key assumptions**

- Treat informational chemistry as a branch point; do not presume a viable RNA replication mechanism.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemistry to heredity` | `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison` | 0 | module=rna-like, temperature=353.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison`**

A reusable heredity-module comparison (Module J). A genuine libRoadRunner/Tellurium SBML ODE integration (needs tellurium) of a quasispecies replicator (Eigen error threshold): a heredity module (RNA-like / peptide / low-fidelity / non-heritable) replicates a polymer with whole-sequence fidelity Q = fidelity^length; above the threshold the functional information is retained and inheritance pays, below it errors win. Compares modules head-to-head on lineage fitness.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = {
    'module'     : 'rna-like',   # Heredity-module preset: rna-like | peptide | low-fidelity | non-heritable
    'temperature': 353.15,   # Temperature (K; ~80 C vent)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `polymer_length` | integer | dimensionless | >= 1 |
| `lineage_fitness` | float | dimensionless |  |
| `information_retention` | float | dimensionless | [0.0, 1.0] |
| `functional_templates` | float | dimensionless | >= 0.0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-06-chemistry-to-heredity ===
STUDY = 'og-06-chemistry-to-heredity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemistry_to_heredity = 100
INTERVAL_Chemistry_to_heredity = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = materialize_composite('viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison['state']}, core=core)
            _comp.run(STEPS_Chemistry_to_heredity)
        print('ran', 'Chemistry to heredity', f'(viva_genesis.composites.heredity_module_comparison.heredity_module_comparison) for', STEPS_Chemistry_to_heredity, 'steps')
    except Exception as _e:
        print(f"Chemistry to heredity not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**heredity-lineage-tree**


In [ ]:
# heredity-lineage-tree
show_viz(_render_one('local:HeredityLineageTree', {'composite': 'viva_genesis.composites.heredity_lineage.heredity_lineage_comparison', 'grid_json': 'workspace/studies/og-06-chemistry-to-heredity/artifacts/heredity_error_threshold_grid.json', 'title': 'Heredity as an inherited lineage'}, RUNS_DB, STUDY_YAML))

**heredity-error-threshold**


In [ ]:
# heredity-error-threshold
show_viz(_render_one('local:HeredityErrorThresholdViz', {'composite': 'viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', 'title': 'Heredity above the error threshold'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| inheritance-beats-noninheritance _(primary)_ | kind=derived path=final_lineage_fitness detail=RNA-like lineage fitness at steady state | op > value 0.1 detail inheriting lineage fitness net-positive (and beats the non-heritable control) |
| information-retained-above-error-threshold _(primary)_ | kind=derived path=final_information_retention detail=C/(C+E) retention at steady state | op >= value 0.4 detail functional information retained (above the error threshold) |
| heredity-modules-compared _(primary)_ | kind=cross_run detail=RNA-like vs peptide lineage fitness | op arms_reported detail both modules viable; RNA-like ranks above peptide |
| error-catastrophe-below-threshold _(diagnostic)_ | kind=cross_run detail=RNA-like vs low-fidelity module | op controls_absent detail the low-fidelity module collapses (retention ~0 fitness negative) None |
| replication-feasible-budget _(supporting)_ | kind=derived path=final_lineage_fitness detail=net fitness after replication cost | op > value 0.0 detail net-positive within the replication budget |
| heredity-lineage-compositional-persists _(primary)_ | kind=cross_run detail=compositional vs non-heritable functional persistence in the agent-based lineage | op arms_reported detail compositional persistence above threshold and above the non-heritable control (realized heritability separates them) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mineral/CO-driven peptide formation at 100 C, pH 7-10 | peptide formation rate/conditions | Huber & Wachtershauser 1998 | Represented as the peptide heredity arm in the agent-based lineage comparison (graded fitness 0.24, below RNA-like 0.38 and the compositional/hybrid arms); a reduced coupling, not a first-principles peptide-formation-rate reproduction |


## Study: Ecology Before LUCA (`og-07-ecology-before-luca`)

**Question.** In connected pore networks with cross-feeding, parasitism, cooperation and horizontal exchange, what emergent capabilities arise — compared to ancestral reconstructions, without assuming LUCA is primitive?

**Objective.** Run heterogeneous populations over a connected pore network; compare emergent capability redundancy (robustness) to the isolated counterfactual, and the collective repertoire to the LUCA reconstruction as a constraint.

**Hypothesis.** Heterogeneous populations develop metabolic diversity, cross-feeding and robustness single compartments do not; the emergent capability set overlaps the anaerobic H2/CO2/FeS LUCA reconstruction as a boundary.

**Claim.** A connected metacommunity of heterogeneous pore compartments develops functional redundancy (robustness) that an isolated compartment cannot — cross-feeding and horizontal transfer spread coarse-grained protometabolic capabilities into >= 2-fold coverage, so most functions survive a single-compartment loss. Robustness EMERGES only in a bounded corner of the swept phase maps — it needs enough inter-compartment diversity to give cross-feeding something to share AND enough exchange to spread it, and it collapses as parasite load rises — too little diversity leaves nothing to share, too little exchange leaves a broad repertoire fragile, and parasites draining their neighbours erase the redundancy. The three-view Decision map over (exchange_rate x parasite_load) at the canonical diversity shows cross-feeding robustness emerging only in the well-connected, low-parasite corner (PASS boundary at robustness 0.5). The collective repertoire's overlap with the anaerobic H2/CO2/FeS LUCA reconstruction is REPORTED as a far-end boundary the emergent set approaches (~0.6-0.7), not a target — LUCA constrains which capabilities are eventually acquired, not evidence the first compartments were LUCA-like.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- ecology: populations + evolution
- LUCA boundary condition

**Proposed models / methods**

- Vivarium multi-agent composites over shared spatial fields
- COBRApy for late-stage genome-scale benchmarks
- evolutionary algorithms

**Key assumptions**

- Use LUCA as a constraint on capabilities eventually acquired, not evidence the first replicators were LUCA-like.
- Each compartment is a coarse-grained capability vector over a protometabolic function set, not an instantiated metabolic network.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Ecology before LUCA` | `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity` | 0 | n_compartments=8, exchange_rate=0.3, diversity=0.75, hgt_rate=0.1, seed=1 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity`**

A reusable cross-feeding metacommunity module (Module L). N connected compartments, each a coarse-grained capability vector over a protometabolic function set, evolving under cross-feeding (byproduct of one feeds the next capability in a neighbour), horizontal transfer, and parasite/cooperator roles. Connected heterogeneous populations develop capability redundancy (robustness) and a collective repertoire single compartments do not; the emergent set's overlap with the anaerobic H2/CO2/FeS LUCA reconstruction is reported as a far-end boundary, not a target. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = {
    'n_compartments': 8,   # Number of connected compartments
    'exchange_rate' : 0.3,   # Cross-feeding strength (use isolated flag, not 0, to disable)
    'diversity'     : 0.75,   # Inter-compartment heterogeneity (0 identical; 1 all different)
    'parasite_load' : 0.0,   # Fraction of compartments that drain neighbours
    'hgt_rate'      : 0.1,   # Horizontal-transfer rate (0 = no spread)
    'isolated'      : False,   # Isolated counterfactual arm: no cross-feeding/HGT/parasitism (use this, not exchange_rate=0)
    'seed'          : 1,   # RNG seed (reproducible assignment)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-07-ecology-before-luca ===
STUDY = 'og-07-ecology-before-luca'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Ecology_before_LUCA = 100
INTERVAL_Ecology_before_LUCA = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = materialize_composite('viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity['state']}, core=core)
            _comp.run(STEPS_Ecology_before_LUCA)
        print('ran', 'Ecology before LUCA', f'(viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity) for', STEPS_Ecology_before_LUCA, 'steps')
    except Exception as _e:
        print(f"Ecology before LUCA not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ecology-three-view**


In [ ]:
# ecology-three-view
show_viz(_render_one('local:EcologyMetacommunityThreeView', {'composite': 'viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', 'grid_json': 'workspace/studies/og-07-ecology-before-luca/artifacts/ecology_robustness_grid.json', 'title': 'Campaign VII — ecology before LUCA'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None)))

**metacommunity-phase-map**


In [ ]:
# metacommunity-phase-map
show_viz(_render_one('local:MetacommunityPhaseMap', {'composite': 'viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', 'grid_json': 'workspace/studies/og-07-ecology-before-luca/artifacts/metacommunity_robustness_grid.json', 'title': 'Where cross-feeding robustness emerges'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cross-feeding-robustness _(primary)_ | kind=derived path=metacommunity_robustness detail=settled metacommunity robustness (functional redundancy) of the connected run | op > value 0.5 detail a majority of capabilities are redundantly covered (robust to a single-compartment loss) |
| connected-beats-isolated _(primary)_ | kind=cross_run detail=connected vs isolated metacommunity robustness across an 8-seed sweep | op arms_reported detail connected robustness strictly higher than isolated on every seed (measured min gain reported) |
| luca-capability-overlap _(supporting)_ | kind=cross_run detail=collective repertoire overlap with the anaerobic LUCA core (Weiss 2016) | op arms_reported detail overlap reported as a boundary not a target None |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA: anaerobic, H2-dependent, CO2-fixing, FeS-rich (355 protein families) | emergent capability overlap | Weiss 2016 / Moody 2024 | Measured — collective repertoire overlaps the anaerobic LUCA core at ~0.6-0.7 (reported boundary) |


## Study: Escape From the Vent (`og-08-escape-from-the-vent`)

**Question.** What is the minimum set of innovations required for autonomous persistence away from the geological reactor, as environmental support is relaxed?

**Objective.** Relax environmental support via ablation sweeps and identify the minimum innovation set for autonomy.

**Hypothesis.** A bounded innovation set (internal energy conversion + resource import + heredity) allows persistence as gradient strength/mineral access/feed are reduced; below it, systems stay obligately vent-tied.

**Claim.** Autonomy away from the geological reactor requires an irreducible innovation set — internal energy conversion (a pump), active resource import, and heredity to hold them. Because the lineage's budget is co-limited (it needs BOTH usable free energy AND usable resources, Liebig's minimum), a single innovation is a cost without a co-benefit; only the full set keeps the net budget above the survival floor as the external gradient, mineral access and feed are all withdrawn. A knockout sweep shows the full set persists off-gradient (surviving fraction 1.0) while removing any one innovation collapses persistence to 0 — so the minimum robust set for escaping the vent is {pump, importer, heredity}.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- all modules integrated
- autonomy readout

**Proposed models / methods**

- multi-agent composites with relaxed boundaries
- ablation sweeps over gradient/mineral/feed
- population genetics

**Key assumptions**

- Autonomy is reduced dependence on the exact vent interface, measured by ablation — not a binary.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Escape from the vent` | `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence`**

A reusable off-gradient-persistence module (Module M). A lineage's co-limited (Liebig-minimum) free-energy + resource budget under relaxing geological support (external gradient, mineral access, feed all declining over the run), with three toggleable innovations: internal energy conversion (has_pump), active resource import (has_importer) and information maintenance (has_heredity). Because the budget is co-limited the innovation set is irreducible — as support -> 0 only the full set keeps net budget above the survival floor, so a knockout sweep identifies {pump, importer, heredity} as the minimum robust set for autonomy. Coarse-grained, deterministic reproduction (not a wrapped ALife engine).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = {
    'gradient_support'   : 1.0,   # Starting external-gradient support, relaxed to 0 over the run (dimensionless, 0-1)
    'mineral_access'     : 1.0,   # Starting mineral / FeS-catalyst access, relaxed to 0 (dimensionless, 0-1)
    'feed'               : 1.0,   # Starting geochemical feed, relaxed to 0 (dimensionless, 0-1)
    'has_pump'           : 1.0,   # Internal energy conversion innovation (1 present; 0 knockout)
    'has_importer'       : 1.0,   # Active resource-import innovation (1 present; 0 knockout)
    'has_heredity'       : 1.0,   # Information-maintenance innovation; without it the machinery degrades (1 present; 0 knockout)
    'catalyst_efficiency': 1.0,   # Internal-pump catalytic efficiency (dimensionless)
    'decline_rate'       : 0.02,   # Support relaxation per step (1/s); support reaches 0 partway through the run
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `gradient_support` | float | dimensionless | [0.0, 1.0] |
| `mineral_access` | float | dimensionless | [0.0, 1.0] |
| `feed` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux >= 0 and co-limited budget (min of energy, resource)`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-08-escape-from-the-vent ===
STUDY = 'og-08-escape-from-the-vent'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Escape_from_the_vent = 100
INTERVAL_Escape_from_the_vent = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = materialize_composite('viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence['state']}, core=core)
            _comp.run(STEPS_Escape_from_the_vent)
        print('ran', 'Escape from the vent', f'(viva_genesis.composites.off_gradient_persistence.off_gradient_persistence) for', STEPS_Escape_from_the_vent, 'steps')
    except Exception as _e:
        print(f"Escape from the vent not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**escape-three-view**


In [ ]:
# escape-three-view
show_viz(_render_one('local:EscapeOffGradientThreeView', {'composite': 'viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', 'grid_json': 'workspace/studies/og-08-escape-from-the-vent/artifacts/off_gradient_grid.json', 'title': 'Campaign VIII — escape from the vent'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None)))

**off-gradient-survival-frontier**


In [ ]:
# off-gradient-survival-frontier
show_viz(_render_one('local:OffGradientPhaseMap', {'composite': 'viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', 'grid_json': 'workspace/studies/og-08-escape-from-the-vent/artifacts/off_gradient_grid.json', 'title': 'The survival frontier off the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| survival-off-gradient _(primary)_ | kind=derived path=offgradient_survival detail=surviving fraction at the end of the run as support -> 0=None | op >= value 0.5 detail a lineage persists off-gradient (surviving fraction >= 0.5) |
| minimum-innovation-set _(primary)_ | kind=cross_run detail=full set vs each single knockout surviving fraction off-gradient=None | op arms_reported detail full set persists off-gradient and every single knockout collapses (each innovation is necessary) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Alkaline-vent origin — the first cells are vent-tied, dependent on the geochemical proton gradient and FeS mineral catalysis; escaping to free-living requires internalising energy generation. | off-gradient persistence vs innovation set | Martin & Russell 2007; Lane & Martin 2012; Sojo et al. 2014 | Reproduced as a coarse-grained co-limited budget — the full innovation set persists off-gradient; each single knockout collapses. |


## Study: The Adversarial Control (`og-adv-generic-nonequilibrium`)

**Question.** Does a generic nonequilibrium chemical system with no vent-specific features (no sustained H2, no Fe/Ni/S catalytic interface, no pore concentration) pass the vent-specific gates? It SHOULD NOT.

**Objective.** Run the og-01..03 behavior tests on a vent-feature-ablated control; the campaign's gates must reject it, and the vent must pass the same tests.

**Hypothesis.** The og-01/og-02/og-03 gates REJECT this control; if they pass it, the gates measure generic nonequilibrium chemistry, not the vent mechanism.

**Claim.** The og-01/02/03 gates are vent-specific and falsifiable. A generic nonequilibrium control — the SAME real composites with no sustained H2, no Fe/Ni/S catalytic interface and no pore concentration — is rejected by every discriminating gate (carbon flux, formate yield, protometabolic throughput, thermophoretic Péclet, enrichment), while the vent passes all five. So a PASS elsewhere in the campaign certifies the vent MECHANISM, not merely "some nonequilibrium chemistry".


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- the same real og-01/02/03 composites with vent features (H2, Fe/Ni/S catalyst, pore concentration) disabled

**Proposed models / methods**

- flat gradient, inert surface, open pore, well-mixed — a generic abiotic nonequilibrium mixture
- re-run the og-01/02/03 behavior-test gates unchanged and require rejection

**Key assumptions**

- This study passes by REJECTING; a control that passes the gates would mean the gates are not vent-specific.
- The control is a parameterization of the real og-01/02/03 composites, so it tests the same models, not a mock.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Generic non-vent control (carbon-fixation composite, vent features off)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.0, catalyst_active=0.0, h2=1e-09 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy finite-volume Nernst-Planck pH-gradient electrochemistry (pmf, current density; needs fipy). Module D = genuine eQuilibrator ΔG' feasibility gate (needs equilibrator_api) + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |
| `formate` | float | molar | >= 0.0 |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-adv-generic-nonequilibrium ===
STUDY = 'og-adv-generic-nonequilibrium'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 100
INTERVAL_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off)
        print('ran', 'Generic non-vent control (carbon-fixation composite, vent features off)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off, 'steps')
    except Exception as _e:
        print(f"Generic non-vent control (carbon-fixation composite, vent features off) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-specificity-three-view**


In [ ]:
# vent-specificity-three-view
show_viz(_render_one('local:VentSpecificityThreeView', {'title': 'Adversarial control — vent-specificity of the gates', 'grid_json': 'workspace/studies/og-adv-generic-nonequilibrium/artifacts/vent_specificity_grid.json'}, RUNS_DB, STUDY_YAML))

**vent-specificity-map**


In [ ]:
# vent-specificity-map
show_viz(_render_one('local:VentSpecificityMap', {'title': 'Adversarial control — vent-specificity of the gates', 'grid_json': 'workspace/studies/og-adv-generic-nonequilibrium/artifacts/vent_specificity_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| gates-reject-generic _(primary)_ | kind=observable detail=run og-01/02/03 gates on the generic control; each measured value below its threshold | op controls_absent detail all five gates reject generic (net flux / formate / throughput / Péclet / enrichment below threshold) |
| vent-specificity-demonstrated _(primary)_ | kind=observable detail=vent vs generic on identical gates | op arms_reported detail vent passes 5/5 where the control fails 5/5 |


## Study: The Nitrogen Bottleneck (`og-res-nitrogen-assimilation`)

**Question.** Carbon fixation is not the only requirement for a protocell. Can an FeS-mediated vent reduce the early ocean's oxidised / inert nitrogen (NO3-/NO2-/N2) to ammonium fast enough to keep pace with carbon fixation, and which conditions (FeS activity, nitrate supply, temperature) decide whether nitrogen becomes limiting first?

**Objective.** Add nitrogen as its own module (FeS-mediated NO3-/NO2- -> NH4+ with minor N2 fixation and carbon-demanded reductive amination), track the C:N supply ratio, and map the region where reduced-N production keeps pace with the carbon-fixation demand.

**Hypothesis.** On an Fe/Ni/S catalyst with an H2 reductant, nitrate is reduced through nitrite to ammonium and drawn into amino-acid precursors by reductive amination; reduced N keeps pace with reduced carbon (C:N supply ratio near Redfield) only inside a bounded region of (nitrate supply x FeS activity), and outside it — low feed or passivated mineral — the C:N ratio blows up and nitrogen, not carbon or energy, becomes the first limiting resource.

**Claim.** FeS-mediated reduction of oxidised/inert nitrogen supplies ammonium and amino-acid precursors fast enough to keep pace with carbon fixation (C:N supply ratio near Redfield) only inside a bounded region of (nitrate supply x FeS activity); outside it nitrogen — not carbon or energy — becomes the first limiting resource.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- I: FeS-mediated nitrate/nitrite reduction to ammonium
- I: minor abiotic N2 fixation on fresh FeS
- I: carbon-demanded reductive amination toward amino-acid precursors

**Proposed models / methods**

- FeS-catalysed dissimilatory nitrate reduction kinetics (H2-gated, temperature-windowed)
- Redfield-like C:N supply-ratio accounting
- Open-flow (feed + washout) sustained-throughput reactor

**Key assumptions**

- Reduced nitrogen (NH4+) is a hard co-requirement with reduced carbon; the early ocean's accessible N is oxidised (NO3-/NO2-) or inert (N2).
- The useful decision observable is the C:N SUPPLY ratio (production vs demand), not any single pool.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Nitrogen as a limiting resource — favourable` | `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor` | 0 | — |
| `Nitrogen-limited (scarce nitrate feed)` | `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor` | 0 | no3_supply=2e-06 |

Declared parameter sets (`study.yaml` variants):

- **nitrogen-limited** — `no3_supply=2e-06`
- **control-no-fes** — `fes_activity=1e-06`
- **control-no-h2** — `h2=1e-12`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor`**

A reusable FeS-mediated reduced-nitrogen module. The real ReducedNitrogenProductionProcess reduces NO3-/NO2- to NH4+ on an Fe/Ni/S catalyst (H2-dependent, temperature-windowed), adds a minor abiotic N2-fixation source, and draws ammonium into amino-acid precursors by reductive amination at a rate set by the carbon-fixation demand. Tracks the C:N supply ratio (carbon demand / reduced-N production) to test whether reduced nitrogen keeps pace with reduced carbon. fes_activity=0 / h2=0 are the no-mineral / no-reductant controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = {
    'no3_supply'  : 2e-05,   # Open-flow nitrate feed (M/s); sets the ceiling on sustained reduced-N production
    'fes_activity': 1.0,   # Active Fe/Ni/S catalyst in [0,1] (0 = no-mineral control)
    'h2'          : 0.005,   # Dissolved H2 reductant (M); ~0 = no-reductant control
    'temperature' : 353.15,   # Temperature (K); FeS nitrate-reduction optimum ~80 C
    'c_demand'    : 0.0001,   # Carbon-fixation demand pulling reduced N into precursors (mol/s)
    'no3_init'    : 0.001,   # Initial nitrate pool (M)
    'n2'          : 0.0005,   # Dissolved N2 available for minor fixation (M)
    'loss_rate'   : 0.01,   # Open-flow washout of NH4+/precursor (1/s); 0 = closed batch (conservation control)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `NO3` | float | molar | >= 0.0 |
| `H2` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `fes_activity` | float | dimensionless | [0.0, 1.0] |
| `c_demand` | float | mole/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `NH4` | float | molar | >= 0.0 |
| `amino_precursor` | float | molar | >= 0.0 |
| `nh4_production_rate` | float | mole/second | >= 0.0 |
| `cn_supply_ratio` | float | dimensionless | >= 0.0 |
| `total_n` | float | molar | >= 0.0 |

**Conditions**

- `nitrogen_mass_balance` (invariant): `abs(delta_total_N - (feed - washout)) < tol`
- `reductant_necessity` (invariant): `nh4_production == 0 when (fes_activity == 0 or H2 == 0)`
- `non_negative_pools` (invariant): `NO3 >= 0 and NO2 >= 0 and NH4 >= 0`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-res-nitrogen-assimilation ===
STUDY = 'og-res-nitrogen-assimilation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Nitrogen_as_a_limiting_resource_favourable = 100
INTERVAL_Nitrogen_as_a_limiting_resource_favourable = 1.0
STEPS_Nitrogen_limited_scarce_nitrate_feed = 100
INTERVAL_Nitrogen_limited_scarce_nitrate_feed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = materialize_composite('viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor['state']}, core=core)
            _comp.run(STEPS_Nitrogen_as_a_limiting_resource_favourable)
        print('ran', 'Nitrogen as a limiting resource — favourable', f'(viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor) for', STEPS_Nitrogen_as_a_limiting_resource_favourable, 'steps')
    except Exception as _e:
        print(f"Nitrogen as a limiting resource — favourable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = materialize_composite('viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor['state']}, core=core)
            _comp.run(STEPS_Nitrogen_limited_scarce_nitrate_feed)
        print('ran', 'Nitrogen-limited (scarce nitrate feed)', f'(viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor) for', STEPS_Nitrogen_limited_scarce_nitrate_feed, 'steps')
    except Exception as _e:
        print(f"Nitrogen-limited (scarce nitrate feed) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**nitrogen-assimilation-timeseries**


In [ ]:
# nitrogen-assimilation-timeseries
show_viz(_render_one('local:NitrogenAssimilationViz', {'composite': 'viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', 'n_steps': 120, 'title': 'Reduced-nitrogen assimilation (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None)))

**c-to-n-supply-balance**


In [ ]:
# c-to-n-supply-balance
show_viz(_render_one('local:NitrogenCarbonBalanceViz', {'composite': 'viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', 'n_steps': 120, 'title': 'Does reduced N keep pace with carbon? (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None)))

**nitrogen-assimilation-phase-map**


In [ ]:
# nitrogen-assimilation-phase-map
show_viz(_render_one('local:NitrogenAssimilationPhaseMap', {'title': 'Nitrogen as a limiting resource (Decision view)', 'grid_json': 'workspace/studies/og-res-nitrogen-assimilation/artifacts/nitrogen_assimilation_phase_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ammonium-production-sustained _(primary)_ | kind=derived path=settled_nh4_uM detail=settled ammonium pool uM=None | op > value 100.0 detail settled NH4+ > 100 uM (a sustained reduced-N pool) |
| reduced-nitrogen-source-exists _(primary)_ | kind=derived path=peak_nh4_production_rate detail=peak ammonium production rate | op > value 1e-06 detail peak NH4+ production > 0 (viable reduced-N source) |
| nitrogen-keeps-pace-with-carbon _(primary)_ | kind=derived path=cn_supply_ratio_settled detail=settled C:N supply ratio | op range low 1.0 high 8.0 detail C:N in the viable keeps-pace band (near Redfield) |
| nitrogen-mass-conserved _(diagnostic)_ | kind=derived path=max_abs_n_mass_residual detail=max |per-step N-mass residual| | op <= value 1e-09 detail max |nitrogen-mass residual| at machine precision |
| feature-ablation-no-ammonium _(primary)_ | kind=cross_run detail=favourable ammonium vs no-FeS / no-H2 | op controls_absent detail ammonium present with FeS+H2 <1 uM in every control None |
| nitrogen-supply-sets-cn _(diagnostic)_ | kind=cross_run detail=favourable C:N vs scarce-nitrate C:N | op arms_reported detail report both arms; flag where nitrogen becomes limiting first |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Ammonia forms from dinitrogen on iron-sulfide surfaces (low yield, fresh-surface dependent) at ~70-80 C | ammonium production on FeS vs controls | Doerr 2003 | Met qualitatively — the FeS-mediated route produces ammonium only with an active mineral and an H2 reductant; N2 fixation is modelled as a deliberately minor source alongside the dominant nitrate-reduction route. |
| Native-metal / mineral catalysis drives reductive prebiotic chemistry toward amino-acid precursors | reduced-N draw into amino-acid precursors | Varma 2018 / Huber 1997 | Met — reductive amination draws ammonium into an amino-acid-precursor pool at a rate set by the carbon-fixation demand (the C:N coupling). |


## Study: The Phosphorus Bottleneck (`og-res-phosphorus-availability`)

**Question.** Phosphorus may be a harder bottleneck than carbon. Can the same thermal-gradient vent pore that concentrates C and N also co-concentrate P — tracking dissolved phosphate, mineral-bound P, polyphosphate and reactive P with explicit precipitation/adsorption loss — or does phosphate precipitate/adsorb before it is chemically useful? Are there local windows where reactive P is produced faster than it is lost?

**Objective.** Build a real P-speciation process (phosphate <-> polyphosphate <-> reactive-P with a precipitation/adsorption mineral sink), couple it to the og-03 thermal pore, and map where reactive-P availability beats loss and co-concentrates with carbon.

**Hypothesis.** In a thermal-gradient pore (coupled from og-03), enrichment drives phosphate condensation and activation steeply enough that reactive-P formation outruns the first-order precipitation/adsorption sink in a bounded (source × enrichment) region — and that same enrichment co-concentrates P alongside carbon; without the gradient (open pore), precipitation wins and no reactive P accumulates.

**Claim.** A real libRoadRunner P-speciation network coupled to the og-03 thermal pore shows reactive phosphorus becomes available only in a bounded (phosphate source × pore enrichment) window where condensation + activation outrun the precipitation/adsorption sink; the same pore enrichment that concentrates carbon co-concentrates P, and without the gradient phosphate precipitates before it is useful — phosphorus is a localization-gated bottleneck, not a free resource.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- I: limiting-resource phosphorus speciation
- F: transport/concentration/retention (og-03 pore coupling)

**Proposed models / methods**

- SBML + libRoadRunner/Tellurium P-speciation network
- og-03 FiPy thermophoretic trap as the enrichment coupling
- P-atom mass balance + precipitation/adsorption sink

**Key assumptions**

- Reactive-P availability arises from the balance of condensation, activation, hydrolysis and precipitation/adsorption — not assumed; the precipitation sink is explicit.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Phosphorus availability: the phosphorus bottleneck` | `viva_genesis.composites.phosphorus_availability.phosphorus_availability` | 0 | phosphate_supply=0.0001, pore_enrichment=30.0, temperature=333.15, loss_rate=0.08 |

Declared parameter sets (`study.yaml` variants):

- **control-open-pore** — `pore_enrichment=1.0`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.phosphorus_availability.phosphorus_availability`**

A reusable phosphorus-availability module. A genuine libRoadRunner/Tellurium SBML ODE integration (needs tellurium) of a P-speciation network (phosphate <-> polyphosphate <-> reactive-P with a precipitation/adsorption mineral sink) driven by a dissolved-phosphate supply, the og-03 thermal-gradient pore enrichment, and pore temperature. Emergent: the standing reactive-P pool, reactive-P production vs precipitation loss (net availability), and the precipitated fraction — does reactive P become available before phosphate precipitates?


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability = {
    'phosphate_supply': 0.0001,   # Dissolved orthophosphate supply / initial pool (M)
    'pore_enrichment' : 30.0,   # Thermal-gradient pore enrichment from the og-03 trap (>=1; 1 = open pore / no gradient control)
    'temperature'     : 333.15,   # Mean pore temperature (K)
    'loss_rate'       : 0.08,   # Precipitation / adsorption loss rate k_prec (1/s) — the mineral-sink bottleneck
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `phosphate_supply` | float | molar | >= 0.0 |
| `pore_enrichment` | float | dimensionless | >= 1.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `reactive_P` | float | molar | >= 0.0 |
| `net_reactive_P` | float | mole/second |  |
| `precipitated_fraction` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `phosphorus_mass_balance` (invariant): `abs(total_P - supplied_P) / supplied_P < tol`
- `nonnegative_pools` (invariant): `phosphate >= 0 and polyphosphate >= 0 and reactive_P >= 0 and mineral_P >= 0`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-res-phosphorus-availability ===
STUDY = 'og-res-phosphorus-availability'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Phosphorus_availability_the_phosphorus_bottleneck = 100
INTERVAL_Phosphorus_availability_the_phosphorus_bottleneck = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_phosphorus_availability_phosphorus_availability = materialize_composite('viva_genesis.composites.phosphorus_availability.phosphorus_availability', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_phosphorus_availability_phosphorus_availability['state']}, core=core)
            _comp.run(STEPS_Phosphorus_availability_the_phosphorus_bottleneck)
        print('ran', 'Phosphorus availability: the phosphorus bottleneck', f'(viva_genesis.composites.phosphorus_availability.phosphorus_availability) for', STEPS_Phosphorus_availability_the_phosphorus_bottleneck, 'steps')
    except Exception as _e:
        print(f"Phosphorus availability: the phosphorus bottleneck not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**phosphorus-speciation**


In [ ]:
# phosphorus-speciation
show_viz(_render_one('local:PhosphorusSpeciationViz', {'composite': 'viva_genesis.composites.phosphorus_availability.phosphorus_availability', 'n_steps': 120, 'title': 'Phosphorus bottleneck — P-species trajectories (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None)))

**phosphorus-mechanism**


In [ ]:
# phosphorus-mechanism
show_viz(_render_one('local:PhosphorusMechanismViz', {'composite': 'viva_genesis.composites.phosphorus_availability.phosphorus_availability', 'n_steps': 120, 'title': 'Phosphorus bottleneck — reactive-P production vs loss (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None)))

**phosphorus-availability-map**


In [ ]:
# phosphorus-availability-map
show_viz(_render_one('local:PhosphorusAvailabilityMap', {'title': 'The phosphorus bottleneck — reactive-P availability landscape (Decision view)', 'grid_json': 'workspace/studies/og-res-phosphorus-availability/artifacts/phosphorus_availability_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reactive-p-available _(primary)_ | kind=derived path=phosphorus_net_reactive_availability detail=settled net reactive-P availability = production - precipitation loss | op > value 0.0 detail net availability > 0 (reactive P produced faster than lost) |
| p-coconcentrates-with-carbon _(primary)_ | kind=cross_run detail=og-03 pore enrichment vs the carbon co-concentration floor; P reactive-availability at that enrichment | op arms_reported detail the same pore regime co-concentrates P with C (enrichment >= carbon floor AND P reactive-availability positive) |
| open-pore-no-enrichment-control _(primary)_ | kind=cross_run detail=favorable enriched pore vs open-pore / no-gradient control | op controls_absent detail reactive-P availability collapses (net <= 0 ~fully precipitated) without enrichment None |
| p-mass-conserved _(diagnostic)_ | kind=derived path=phosphorus_p_mass_residual detail=max |total P - supplied| / supplied | op <= value 1e-06 detail P-mass residual small (P-atom-conserving network) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Reactive reduced-P (schreibersite-derived) and condensed/activated phosphates form under basic conditions; dissolved phosphate is prone to precipitation/adsorption. | reactive-P availability vs precipitated fraction | Pasek 2017 | measured — reactive-P availability + precipitated fraction graded from the real run |


## Study: The Energy Currency (`og-energy-proto-energetics`)

**Question.** Can a primitive system capture environmental (geochemical) free energy into a transferable chemical currency — charged by an exergonic drive, spent on endergonic work, lost to hydrolysis — before any modern-style ion pumping, and what fraction of the geochemical free energy becomes stored usable work?

**Objective.** Represent the exergonic-to-endergonic coupling as explicit energetic bookkeeping (a free-energy ledger), not a favourable reaction direction; map where a transferable currency is maintained and measure the stored-work fraction.

**Hypothesis.** A pre-ATP carrier (thioester / acyl-phosphate class) coupled to an exergonic geochemical drive maintains a transferable currency above threshold under endergonic load — but only where the drive ΔG clears the carrier's high-energy bond and the coupling is efficient enough to beat discharge + hydrolysis. A bounded fraction of the geochemical free energy is captured as stored usable work; the rest is dissipated. Remove the coupling or starve the drive below the bond and the currency collapses.

**Claim.** A pre-ATP thioester-class carrier coupled to an exergonic geochemical drive maintains a transferable energy currency under endergonic load (charged fraction ~0.39) and captures ~47% of the tapped geochemical free energy as stored usable work; removing the coupling or starving the drive below the carrier bond collapses the currency — capturing free energy into a transferable store is an explicit, bounded, falsifiable transition, not a favourable reaction direction.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- energetics: pre-ATP energy currency (thioesters / acyl phosphates / polyphosphates)

**Proposed models / methods**

- free-energy ledger (input = useful work + stored + dissipated) as a conservation invariant
- charge / discharge / hydrolysis kinetics of a transferable carrier pool
- phase map over (geochemical free-energy drive x coupling efficiency)

**Key assumptions**

- Capturing environmental free energy into a transferable store is the transition — not merely a favourable reaction direction.
- A carrier can only be charged when the drive ΔG clears its high-energy bond (a thermodynamic gate, not just a rate).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Proto-energetics (favourable energy currency)` | `viva_genesis.composites.proto_energetics.proto_energetics` | 0 | drive_free_energy=50.0, coupling_efficiency=0.6, carrier_pool=0.001, loss_rate=0.05, load=0.2, carrier=thioester |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.proto_energetics.proto_energetics`**

A transferable energy currency before modern pumping (pre-og-05). An exergonic geochemical drive charges a carrier pool (thioester / acyl-phosphate class) that is discharged for endergonic work under load and bled by hydrolysis, all tracked in a conserved free-energy ledger (input = work + stored + dissipated). Tests whether the system captures environmental free energy into a transferable store, and what fraction becomes stored usable work. drive_free_energy / coupling_efficiency set the arm; coupling->0 or drive below the carrier bond collapses the currency (ablations). Coarse-grained, deterministic reproduction (not a wrapped engine).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_proto_energetics_proto_energetics = {
    'drive_free_energy'  : 50.0,   # Exergonic geochemical drive ΔG available (kJ/mol); below the carrier bond the drive cannot charge it
    'coupling_efficiency': 0.6,   # Fraction of tapped drive free energy coupled into the carrier bond (0 = no coupling ablation)
    'carrier_pool'       : 0.001,   # Total carrier concentration (molar), charged + free
    'loss_rate'          : 0.05,   # First-order hydrolysis / loss rate of charged carrier (per tick)
    'load'               : 0.2,   # Endergonic work demand as a fraction of the carrier pool per tick
    'carrier'            : 'thioester',   # Primitive carrier chemistry (thioester / acyl_phosphate / polyphosphate / reduced_metal)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `drive_free_energy` | float | kilojoule/mole | >= 0.0 |
| `coupling_efficiency` | float | dimensionless | [0.0, 1.0] |
| `load` | float | dimensionless | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `energy_charge` | float | molar | >= 0.0 |
| `energy_charge_fraction` | float | dimensionless | [0.0, 1.0] |
| `energy_input` | float | kilojoule/liter | >= 0.0 |
| `energy_work` | float | kilojoule/liter | >= 0.0 |
| `energy_stored` | float | kilojoule/liter | >= 0.0 |
| `energy_dissipated` | float | kilojoule/liter | >= 0.0 |
| `energy_stored_work_fraction` | float | dimensionless | [0.0, 1.0] |
| `energy_ledger_residual` | float | kilojoule/liter |  |

**Conditions**

- `free_energy_accounting` (invariant): `abs(energy_input - (energy_work + energy_stored + energy_dissipated)) < tol`
- `carrier_bounds` (invariant): `0 <= energy_charge <= carrier_pool`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-energy-proto-energetics ===
STUDY = 'og-energy-proto-energetics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Proto_energetics_favourable_energy_currency = 100
INTERVAL_Proto_energetics_favourable_energy_currency = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_proto_energetics_proto_energetics if 'ov_viva_genesis_composites_proto_energetics_proto_energetics' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_proto_energetics_proto_energetics = materialize_composite('viva_genesis.composites.proto_energetics.proto_energetics', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_proto_energetics_proto_energetics['state']}, core=core)
            _comp.run(STEPS_Proto_energetics_favourable_energy_currency)
        print('ran', 'Proto-energetics (favourable energy currency)', f'(viva_genesis.composites.proto_energetics.proto_energetics) for', STEPS_Proto_energetics_favourable_energy_currency, 'steps')
    except Exception as _e:
        print(f"Proto-energetics (favourable energy currency) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**proto-energetics-world**


In [ ]:
# proto-energetics-world
show_viz(_render_one('local:ProtoEnergeticsWorldViz', {'composite': 'viva_genesis.composites.proto_energetics.proto_energetics', 'title': 'Proto-energetics — charging and spending the currency (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_proto_energetics_proto_energetics if 'ov_viva_genesis_composites_proto_energetics_proto_energetics' in dir() else None)))

**proto-energetics-provenance**


In [ ]:
# proto-energetics-provenance
show_viz(_render_one('local:ProtoEnergeticsProvenanceViz', {'title': 'Proto-energetics — the energy-provenance ledger (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**proto-energetics-map**


In [ ]:
# proto-energetics-map
show_viz(_render_one('local:ProtoEnergeticsMap', {'composite': 'viva_genesis.composites.proto_energetics.proto_energetics', 'grid_json': 'workspace/studies/og-energy-proto-energetics/artifacts/proto_energetics_phase_grid.json', 'title': 'Proto-energetics — capturing free energy into a transferable currency (Decision view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_proto_energetics_proto_energetics if 'ov_viva_genesis_composites_proto_energetics_proto_energetics' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| currency-maintained _(primary)_ | kind=derived path=energy_final_charge_fraction detail=charged fraction of the carrier pool at the end of the run under load=None | op >= value 0.25 detail the currency is maintained (charged fraction >= 0.25) against discharge + hydrolysis |
| captures-free-energy _(primary)_ | kind=derived path=energy_stored_work_fraction detail=(delivered work + currently stored)/input free energy at the end of the run | op >= value 0.3 detail capture into transferable currency exceeds 30% of the tapped geochemical free energy |
| ablation-no-coupling-or-drive-collapses _(primary)_ | kind=cross_run detail=no-coupling and no-drive controls vs the favourable arm | op controls_absent detail both ablations collapse (charged fraction ~0) where the favourable arm maintains the currency |
| free-energy-conserved _(supporting)_ | kind=derived path=energy_ledger_residual detail=worst-case |input - (work + stored + dissipated)| over the run | op <= value 1e-09 detail the ledger residual stays at machine precision (free-energy accounting invariant) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Pre-ATP carriers (thioesters, acyl phosphates) couple exergonic to endergonic chemistry as a transferable energy currency before membrane bioenergetics / ATP | charged fraction + stored-work fraction vs drive and coupling | Buckel & Thauer 2013 (flavin-based electron bifurcation / energy coupling); Lane & Martin 2010 (chemiosmotic coupling at the vent) | Reproduced as explicit free-energy bookkeeping (coarse-grained reduced kinetics) |


## Study: Growth and Division (`og-growth-division`)

**Question.** Does an internal autocatalytic/metabolic chemistry change a compartment's PHYSICAL fate — accumulating product, generating osmotic pressure, drawing in solvent, expanding, and forcing division — and under what permeability regime does it divide rather than lyse or starve?

**Objective.** Couple a reduced autocatalytic osmolyte source to compartment osmotic physics (osmotic pressure → solvent influx → volume → division) with a semi-permeable membrane; map the (network throughput × membrane permeability) plane and locate the viable growth→division band between starvation and lysis, conserving mass across growth and division.

**Hypothesis.** Product accumulation from an internal autocatalytic source generates osmotic pressure that drives solvent influx and volume growth to a division threshold; there is a bounded membrane-permeability band in which the compartment sustains net growth leading to division, flanked by starvation (too permeable, the product leaks before osmotic pressure can build) and lysis (too impermeable, the pressure bursts the membrane).

**Claim.** A protocell's chemistry can change its physical fate. An internal autocatalytic/metabolic source makes an osmolyte; its accumulation raises the internal osmotic pressure (van 't Hoff), that pressure draws solvent across the semi-permeable membrane, the compartment expands, and at a size threshold it divides — a physical selection mechanism with no abstract fitness score. Whether this produces division rather than lysis or starvation is set by the membrane permeability against the network throughput — there is a bounded viable band in which net growth leads to repeated division, flanked by starvation (too permeable — the product leaks before the osmotic pressure can build) and lysis (too impermeable — the pressure bursts the membrane). Mass is conserved to machine precision across growth and division.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H+: metabolism changes the compartment's physical fate (osmotic growth → division)
- osmotic pressure as a physical selection mechanism (no abstract fitness score)

**Proposed models / methods**

- custom coarse-grained osmotic membrane kinetics (van 't Hoff + solubility-diffusion permeability)
- reduced autocatalytic source (og-02 rTCA-style turn)
- parameter sweep over (throughput × permeability) → decision surface

**Key assumptions**

- The compartment (og-04 protocell) and the autocatalytic network (og-02 rTCA source) are given; this stage couples them to osmotic physics to test whether chemistry drives a physical division.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Metabolism to osmotic growth and division (favorable)` | `viva_genesis.composites.osmotic_growth_division.osmotic_growth_division` | 0 | network_throughput=1e-19, membrane_permeability=1e-09, autocatalytic_rate=0.5, division_threshold=6.4e-17, leak_rate=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.osmotic_growth_division.osmotic_growth_division`**

A reusable osmotic growth-and-division module (metabolism → growth & division). An internal autocatalytic osmolyte source coupled to compartment osmotic physics: product accumulation → osmotic pressure (van 't Hoff) → solvent influx → volume growth → division at a size threshold. Membrane permeability sets the fate: too permeable starves (product leaks, no osmotic build-up), too impermeable lyses (pressure bursts the membrane), matched gives sustained growth → division. Units-typed, mass-conserving coarse-grained reproduction (not wrapped membrane MD).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division = {
    'network_throughput'   : 1e-19,   # Internal autocatalytic/metabolic osmolyte source rate (mole/second); ~0 = ablation control
    'autocatalytic_rate'   : 0.5,   # Self-amplification gain of the source (dimensionless)
    'membrane_permeability': 1e-09,   # Membrane solute permeability (meter/second); the fate knob (leaky→starve, tight→lyse)
    'division_threshold'   : 6.4e-17,   # Volume at which the compartment divides (liter); default 1.6× initial volume
    'leak_rate'            : 1.0,   # Multiplier on the solute permeability for the outward leak (dimensionless)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `network_throughput` | float | mole/second | >= 0.0 |
| `membrane_permeability` | float | meter/second | >= 0.0 |
| `external_osmolyte` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `volume` | float | liter | >= 0.0 |
| `internal_concentration` | float | molar | >= 0.0 |
| `osmotic_pressure` | float | pascal | >= 0.0 |
| `division_count` | integer | dimensionless | >= 0 |
| `osmolyte_balance_residual` | float | mole |  |

**Conditions**

- `mass_balance` (invariant): `abs(osmolyte_balance_residual) < tol`
- `osmotic_coupling_directionality` (invariant): `osmotic_pressure >= 0 and solvent influx follows the osmotic gradient (dV/dt = Lp*A*Pi)`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-growth-division ===
STUDY = 'og-growth-division'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Metabolism_to_osmotic_growth_and_division_favorable = 100
INTERVAL_Metabolism_to_osmotic_growth_and_division_favorable = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division = materialize_composite('viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division['state']}, core=core)
            _comp.run(STEPS_Metabolism_to_osmotic_growth_and_division_favorable)
        print('ran', 'Metabolism to osmotic growth and division (favorable)', f'(viva_genesis.composites.osmotic_growth_division.osmotic_growth_division) for', STEPS_Metabolism_to_osmotic_growth_and_division_favorable, 'steps')
    except Exception as _e:
        print(f"Metabolism to osmotic growth and division (favorable) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**osmotic-growth-world**


In [ ]:
# osmotic-growth-world
show_viz(_render_one('local:OsmoticGrowthWorld', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'title': 'Growth & division — volume, concentration, division events'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

**osmotic-coupling-mechanism**


In [ ]:
# osmotic-coupling-mechanism
show_viz(_render_one('local:OsmoticCouplingMechanism', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'title': 'The osmotic coupling — why too permeable starves, too impermeable lyses'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

**growth-division-map**


In [ ]:
# growth-division-map
show_viz(_render_one('local:GrowthDivisionMap', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'grid_json': 'workspace/studies/og-growth-division/artifacts/growth_division_grid.json', 'title': 'Metabolism → growth & division — the viable band'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| net-growth-to-division _(primary)_ | kind=derived path=growth_division_count detail=cumulative division count in the favorable (matched) arm | op > value 0.0 detail the compartment divides (>0) in the matched throughput/permeability regime |
| mass-conserved-across-division _(primary)_ | kind=derived path=osmotic_balance_residual_max detail=worst-case |osmolyte mass-balance residual| over the run (mol) | op < value 1e-18 detail mass conserved across the growth/division cycle (residual below a negligible mole tolerance) |
| division-not-lysis-or-starvation _(primary)_ | kind=cross_run detail=matched (divide) vs too-permeable (starve) and too-impermeable (lyse) arms | op arms_reported detail matched divides; too permeable = starve (no division no growth); too impermeable = lyse (viable band is bounded) None |
| ablation-no-throughput-no-growth _(primary)_ | kind=cross_run detail=favorable (source on) vs zero-throughput ablation (source off) | op controls_absent detail favorable divides; zero-throughput ablation shows no division and no net growth |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Internal product accumulation generates osmotic pressure that drives compartment growth and division; fatty-acid / mixed-amphiphile vesicles grow and divide under osmotic and physico-chemical forcing without dedicated division machinery. | viable growth→division band over (throughput × permeability) | Jordan 2019 (vesicle feasibility); Muchowska 2019 (autocatalytic rTCA source); osmotic-growth vesicle division (Nat. Chem. 2023, pending reference) | Reproduced as a coarse-grained, units-typed, mass-conserving osmotic-coupling model — the matched regime grows and divides; the band is bounded by starvation and lysis. |


## Study: Calibrating the Vent: What Fraction of Observed-Consistent Vent States Feed Downstream Chemistry (`og-cal-vent-environment`)

**Question.** Model Upgrade plan, Phase 1 (priority upgrade A). og-00 showed the real PHREEQC + FiPy vent settles in the Lost City window over ~80% of a swept grid — but the fraction of a chosen grid that passes is not a probability of feasibility, and sweep bounds can determine apparent robustness. Conditioning on observed systems instead: of the vent states consistent with the Lost City field observations (outlet T, pH, dissolved H2), what fraction also satisfies what og-01's carbon fixation needs (H2 >= 1 mM at the interface, a >= 3-unit pH gradient, T inside validity)? Which observation binds, which parameters does it pin, and does the answer depend on whether the vent wall convects?

**Objective.** Run the Phase-1 environmental-calibration pipeline on the real geochemical world: LHS prior from the vg-data package vent_environment -> ensemble per spatial-transport model class -> ABC conditioning on the field windows -> posterior-predictive fraction meeting the downstream requirements -> Morris screening -> the Feasibility / Robustness / Discrimination report with the ledger verified.

**Hypothesis.** A Lost-City-consistent posterior exists inside the empirically motivated prior of the og-00 world, most of it also satisfies the downstream requirements, the result is the same under the convecting and the prescribed-upflow transport classes, and the conservation ledger closes exactly on every draw.

**Claim.** Conditioning the real vent on field observations turns og-00's "80% of a swept grid is habitable" into a posterior statement: 12.5% of the empirically motivated prior reproduces Lost City, the dissolved-H2 observation does all the work (pH is buffered, T is loose), and 100% of that posterior supplies the H2 and pH gradient downstream carbon fixation requires — a result that does not depend on whether the wall convects, and that identifies the DIC prior and the equilibrium-vs-kinetic carbon-redox assumption as the decisions that move it.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: serpentinization as boundary flux (PHREEQC speciation)
- B: flow/mixing (convecting vs prescribed-upflow transport classes)

**Proposed models / methods**

- vg-data prior package (value-or-distribution + provenance class per parameter)
- Latin-hypercube ensemble through the real PHREEQC + FiPy geochemical_world per model class
- ABC rejection on field observation windows; posterior-predictive requirement mask
- Morris elementary-effects screening; paired class differences; conservation ledger

**Key assumptions**

- The prior is the vg-data package vent_environment (six free parameters; provenance audit: 0 measured, 2 calibrated, 1 inferred, 3 extrapolated, 2 hypothetical).
- Uniform likelihood on observation windows (ABC rejection): a vent state is consistent when every readout lies inside its window.
- Full redox equilibrium each step (PHREEQC, llnl.dat): DIC is reduced before free H2 appears; a kinetic carbon-redox class is the planned alternative.
- Downstream requirements are og-01's favorable regime (H2 >= 1 mM, delta pH >= 3 against pH 6.0, T <= 100 C), classed inferred.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Geochemical world (Phase 1 operating point)` | `viva_genesis.composites.geochemical_world.geochemical_world` | 0 | — |

Declared parameter sets (`study.yaml` variants):

- **uniform-upflow-transport** — `model_spatial_transport=fipy_uniform_upflow`
- **reduced-serpentinization** — `model_serpentinization=reduced`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.geochemical_world.geochemical_world`**

og-00's real serpentinizing vent (genuine PHREEQC peridotite speciation feeding a genuine FiPy finite-volume advection-diffusion solve) rebuilt as the Phase-1 geochemical world: both mechanisms are selected through typed faces (model_serpentinization: phreeqc | reduced; model_spatial_transport: fipy convecting | fipy_uniform_upflow | fenics), so the model CLASS is a categorical sweep axis beside the physical parameters, and a ConservationLedgerProcess audits integrated sources against accumulated pools, charge balance, flux divergence and heat directionality (ledger/ledger_ok). The free parameters are the vg-data package 'vent_environment' (value-or-distribution, units, context, source, provenance class), which the environmental-calibration ensemble samples.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_geochemical_world_geochemical_world = {
    'model_serpentinization' : 'phreeqc',   # Model class behind the 'serpentinization' face: phreeqc | reduced
    'model_spatial_transport': 'fipy',   # Model class behind the 'spatial_transport' face: fenics | fipy | fipy_uniform_upflow
    'flow_velocity'          : 0.001,   # Boundary water flux through the reaction zone (m/s) [vg-data: inferred]
    'olivine_rate'           : 0.15,   # Olivine reaction extent per flow-saturated step [vg-data: hypothetical]
    'pyroxene_fraction'      : 0.5,   # Diopside per mole olivine (peridotite vs dunite) [vg-data: calibrated]
    'fe_fraction'            : 0.25,   # Fayalite fraction of the dissolving olivine [vg-data: extrapolated]
    'heat_to_fluid_fraction' : 0.0008,   # Serpentinization enthalpy retained in the fluid [vg-data: calibrated]
    'fluid_dic'              : 0.002,   # Inflowing-fluid DIC, Hadean-ocean proxy (molar) [vg-data: extrapolated]
    'nx'                     : 40,   # FiPy finite-volume cells across the interface column
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `olivine_rate` | float | dimensionless | >= 0.0 |
| `fluid_dic` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `H2_flux` | float | mole/second | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |

**Conditions**

- `mass_balance` (invariant): `ledger.H2_residual == 0 and ledger.alkalinity_residual == 0`
- `charge_neutrality` (invariant): `abs(charge_imbalance) < 1e-6`
- `spatial_flux_conservation` (invariant): `flux_divergence_residual < 1e-3`
- `energy_directionality` (invariant): `ledger.energy_directionality == 1`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-cal-vent-environment ===
STUDY = 'og-cal-vent-environment'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Geochemical_world_Phase_1_operating_point = 100
INTERVAL_Geochemical_world_Phase_1_operating_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_geochemical_world_geochemical_world if 'ov_viva_genesis_composites_geochemical_world_geochemical_world' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_geochemical_world_geochemical_world = materialize_composite('viva_genesis.composites.geochemical_world.geochemical_world', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_geochemical_world_geochemical_world['state']}, core=core)
            _comp.run(STEPS_Geochemical_world_Phase_1_operating_point)
        print('ran', 'Geochemical world (Phase 1 operating point)', f'(viva_genesis.composites.geochemical_world.geochemical_world) for', STEPS_Geochemical_world_Phase_1_operating_point, 'steps')
    except Exception as _e:
        print(f"Geochemical world (Phase 1 operating point) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-environment-posterior**


In [ ]:
# vent-environment-posterior
show_viz(_render_one('local:VentEnvironmentPosterior', {'title': 'Phase 1 — Lost-City-conditioned vent posterior (Decision view)', 'ensemble_json': 'workspace/studies/og-cal-vent-environment/artifacts/vent_environment_ensemble.json', 'report_json': 'workspace/studies/og-cal-vent-environment/artifacts/vent_environment_report.json'}, RUNS_DB, STUDY_YAML))

**vent-reactor-timeseries**


In [ ]:
# vent-reactor-timeseries
show_viz(_render_one('local:VentReactorTimeseries', {'composite': 'viva_genesis.composites.geochemical_world.geochemical_world', 'n_steps': 120, 'title': 'Geochemical world at the operating point (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_geochemical_world_geochemical_world if 'ov_viva_genesis_composites_geochemical_world_geochemical_world' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ledger-closes _(primary)_ | kind=derived path=max_ledger_mass_residual detail=worst-case |pool - integral| / integral over H2 and alkalinity | op <= value 1e-12 detail matter ledger residual at machine precision |
| energy-ledger-closes _(primary)_ | kind=derived path=max_ledger_energy_residual detail=worst-case |heat pool - integrated heat| / integral | op <= value 1e-12 detail energy ledger residual at machine precision |
| heat-released-not-absorbed _(diagnostic)_ | kind=derived path=energy_directionality_final detail=1 while the heat source has never been negative | op >= value 1.0 detail directionality witness holds through the run |
| charge-conserved _(diagnostic)_ | kind=derived path=max_abs_charge_imbalance detail=worst-case |charge imbalance| over the run | op <= value 1e-06 detail |charge imbalance| < 1e-6 throughout |
| dissolved-h2-in-lost-city-range _(primary)_ | kind=derived path=settled_dissolved_h2_mM detail=settled H2_flux / (flow_velocity x flux_area) mmol/kg=None | op range low 1.0 high 15.0 detail dissolved H2 in [1, 15] mmol/kg |
| posterior-viable-vent-fraction _(primary)_ | kind=derived path=posterior_viable_fraction detail=viable | consistent, headline transport class, from artifacts/vent_environment_report.json | op > value 0.0 detail posterior-viable fraction > 0 |
| structure-agreement-across-transport-classes _(primary)_ | kind=derived path=structure_agreement_viable detail=fraction of identical draws on which every class agrees about viability | op >= value 0.8 detail cross-class agreement >= 0.8 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Lost City end-member fluids 40-91 C, pH 9-10.8, dissolved H2 1-15 mmol/kg | settled outlet T, pH; dissolved H2 backed out of the H2 flux | Kelley 2001 / Kelley 2005 / Ludwig 2006 / Proskurowski 2008 | Used as the ABC conditioning windows (vg-data vent_environment.observations); 6/48 prior draws reproduce all three. |


## Study: The Vent Wall as a Galvanic Cell: Which Limitation Stops Carbon Fixation, and Do the Model Classes Agree? (`og-01-cal-electrochemistry`)

**Question.** Model Upgrade plan, Phase 2 (electrochemical carbon fixation). og-01 drives CO2 reduction with a scalar gradient_efficiency and a Michaelis drive in the proton-motive force, so it can say WHETHER fixation runs but not WHY it stops. Treating the vent wall as an electrical circuit instead — fluid redox state -> Nernst electrode potentials -> Butler-Volmer overpotentials -> current density -> surface rates — which limitation (thermodynamic, electrical, catalytic, transport) binds where over (vent pH x dissolved H2 x mineral conductivity x Ni/Fe), and do the empirical and the electrochemical model classes on the new carbon_fixation face agree on that label in the region both call favorable?

**Objective.** Build the carbon_fixation face with three classes (empirical = og-01's kinetics; butler_volmer = Nernst + Butler-Volmer galvanic cell, the primary new class; microkinetic = CatMAP/Cantera, pixi-only), the MineralConductivityProcess that derives gradient_efficiency from circuit physics, the ChargeLedgerProcess (Kirchhoff residual per tick, faradaic efficiency <= 1, current-voltage residual) and the vg-data package electrochemical_interface; sweep (pH x H2 x conductivity x Ni) x classes in open flow; attribute the limitation at every point; measure cross-class agreement; report Feasibility / Robustness / Discrimination.

**Hypothesis.** Explicit electrochemistry reproduces the Altair-2025 current window at the nominal operating point with the Preiner product hierarchy, derives og-01's gradient efficiency from the wall's conductivity (an insulating wall collapses it, the Tutolo-2026 arm, as a consequence rather than a knob), closes a charge + electron ledger exactly, and labels every swept point with its dominant limitation. Pre-registered gate: the classes agree on the label on >= 80% of the region both call favorable.

**Claim.** Explicit electrochemistry is feasible, calibrated and ledger-exact, and it changes the answer to "why does fixation stop": under the Butler-Volmer class the alkaline-vent cell is activation-limited everywhere the wall conducts — the catalyst, not the gradient, binds — and the Tutolo-style collapse is an electrical regime below ~1e-6 S/m that both classes recover. The empirical class cannot make that attribution because its one Michaelis drive term lumps electrical and catalytic limitation and its substrate Michaelis constant invents a transport limit the cell physics does not have. The failure-mode label is therefore structure-sensitive (agreement 0.47 < 0.8): a negative result on the gate and the Phase-2 mechanism map the plan asked for, recorded as a decision for the expert rather than hidden.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: empirical gated kinetics (og-01) with the gradient efficiency derived from the wall circuit
- B: the wall as a galvanic cell — Nernst potentials, Butler-Volmer kinetics, ohmic mineral + ionic path, faradaic split
- C: adsorbate microkinetics with site coverage and passivation (deferred; pixi)

**Proposed models / methods**

- carbon_fixation face with the model class as a categorical sweep axis
- Nernst + Butler-Volmer galvanic-cell solve (Brent) per tick; Koutecky-Levich transport limits; Bruggeman ionic path
- Loss-fraction partition of the fresh-fluid emf -> limitation label per point; cross-class agreement + confusion
- Charge + electron ledger (Kirchhoff residual, faradaic efficiency <= 1, current-voltage residual)

**Key assumptions**

- Steady galvanic cell each tick: the CO2/HCOOH couple sets the cathode emf (acetate and pyruvate are side branches of the same current); H2 and CO2 are boundary feeds; standard potentials at 25 C.
- Exchange currents calibrated to the Altair current window, not to per-condition curves; faradaic efficiency set by the competing H2-evolution branch and proton transport at pH 6.
- The empirical class is og-01's rate law unchanged, with the catalyst scaled by the same Ni gain and the gradient efficiency derived from the circuit.
- Favorable region = throughput >= 10% of each class's own maximum; label = largest loss fraction (FE < 0.5 -> catalytic).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Electrochemical interface — Butler-Volmer, open flow` | `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation` | 0 | model_carbon_fixation=butler_volmer, dilution_rate=2.0 |

Declared parameter sets (`study.yaml` variants):

- **empirical-class** — `model_carbon_fixation=empirical`, `dilution_rate=2.0`
- **microkinetic-class-pixi-only** — `model_carbon_fixation=microkinetic`, `dilution_rate=2.0`
- **insulating-wall** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `conductivity=1e-09`
- **iron-only-catalyst** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `ni_fraction=0.0`
- **control-no-catalyst** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `catalyst_active=0.0`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation`**

og-01's H2/pH-driven carbon fixation rebuilt as the Phase-2 electrochemical interface: the carbon-fixation mechanism is a model CLASS on the carbon_fixation face (model_carbon_fixation: empirical | butler_volmer | microkinetic). The empirical class is og-01's lab-calibrated gated kinetics driven by the real FiPy Nernst-Planck proton gradient + proton budget, with gradient_efficiency DERIVED from the wall's circuit (MineralConductivityProcess: conductivity, thickness, porosity vs the electrodes' charge-transfer resistance) instead of a knob. The butler_volmer class solves the vent wall as a galvanic cell each tick (Nernst half-cell potentials from the fluid redox state, Butler-Volmer kinetics scaled by catalyst activity / Ni fraction / T, Koutecky-Levich transport limits, the ohmic mineral + ionic path, the faradaic split over formate / acetate / pyruvate calibrated to the Altair-2025 current window and the Preiner-2020 yield hierarchy). The microkinetic class (CatMAP/Cantera) builds here and runs under pixi. Every class publishes the thermodynamic / electrical / catalytic / transport loss fractions and a limitation code, and a ChargeLedgerProcess audits its electron balance (Kirchhoff residual, faradaic efficiency <= 1, current-voltage residual -> ledger/charge_ledger_ok). Free parameters are the vg-data package 'electrochemical_interface'.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = {
    'model_carbon_fixation': 'butler_volmer',   # Model class behind the 'carbon_fixation' face: butler_volmer | empirical | microkinetic
    'ph_vent'              : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'             : 6.0,   # Acidic early-ocean-side pH
    'h2'                   : 0.005,   # Dissolved H2 feed, vent side (M)
    'co2'                  : 0.002,   # Dissolved CO2 feed, ocean side (M)
    'temperature'          : 373.15,   # Interface temperature (K)
    'conductivity'         : 0.001,   # Electronic conductivity of the mineral layer (S/m) [vg-data: extrapolated]
    'layer_thickness'      : 0.001,   # Mineral layer thickness (m) [vg-data: inferred]
    'ni_fraction'          : 0.75,   # Ni mole fraction of the Fe/Ni catalytic phase (0 greigite .. 0.75 awaruite) [vg-data: measured]
    'porosity'             : 0.35,   # Wall porosity (ionic return path)
    'catalyst_active'      : 1.0,   # Active Fe/Ni/S mineral catalyst fraction (0 = no-catalyst control)
    'j0_co2'               : 1e-05,   # CO2-reduction exchange current density at Ni3Fe, 100 C (A/m^2) [vg-data: calibrated]
    'boundary_layer'       : 0.001,   # Diffusion boundary layer at the electrodes (m) [vg-data: inferred]
    'dilution_rate'        : 0.0,   # Open-flow washout (1/s); 0 = closed batch, >0 = sustained-throughput regime
    'nx'                   : 20,   # FiPy cells across the Nernst-Planck barrier
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `ph_vent` | float | dimensionless | [0.0, 14.0] |
| `h2` | float | molar | >= 0.0 |
| `conductivity` | float | siemens/meter | >= 0.0 |
| `ni_fraction` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second | >= 0.0 |
| `current_density` | float | ampere/meter**2 | >= 0.0 |
| `faradaic_efficiency` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `charge_conservation` (invariant): `ledger.charge_residual <= 1e-9 every tick`
- `faradaic_bound` (invariant): `ledger.faradaic_efficiency <= 1`
- `current_voltage_consistency` (invariant): `ledger.iv_residual_max <= 1e-6`
- `loss_partition` (invariant): `loss_thermodynamic + loss_electrical + loss_catalytic + loss_transport == 1 while current flows (butler_volmer)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-cal-electrochemistry ===
STUDY = 'og-01-cal-electrochemistry'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Electrochemical_interface_Butler_Volmer_open_flow = 100
INTERVAL_Electrochemical_interface_Butler_Volmer_open_flow = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation if 'ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = materialize_composite('viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Electrochemical_interface_Butler_Volmer_open_flow)
        print('ran', 'Electrochemical interface — Butler-Volmer, open flow', f'(viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation) for', STEPS_Electrochemical_interface_Butler_Volmer_open_flow, 'steps')
    except Exception as _e:
        print(f"Electrochemical interface — Butler-Volmer, open flow not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**electrochemistry-mechanism-map**


In [ ]:
# electrochemistry-mechanism-map
show_viz(_render_one('local:ElectrochemistryMechanismMap', {'title': 'Phase 2 — electrochemical mechanism map (Decision view)', 'mechanism_map_json': 'workspace/studies/og-01-cal-electrochemistry/artifacts/electrochemistry_mechanism_map.json', 'report_json': 'workspace/studies/og-01-cal-electrochemistry/artifacts/electrochemistry_report.json'}, RUNS_DB, STUDY_YAML))

**electrochemistry-polarization**


In [ ]:
# electrochemistry-polarization
show_viz(_render_one('local:ElectrochemistryPolarization', {'title': 'Vent wall as a galvanic cell at the nominal point (Mechanism view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| failure-mode-labelled _(primary)_ | kind=derived path=failure_mode_label_coverage detail=min over classes of the fraction of points with a finite label | op >= value 1.0 detail label coverage 1.0 in every class |
| failure-mode-attributed _(primary)_ | kind=derived path=failure_mode_agreement_favorable detail=fraction of favorable-both points with identical labels | op >= value 0.8 detail cross-class label agreement >= 0.8 in the favorable region |
| mechanism-map-ledger-closes _(primary)_ | kind=derived path=mechanism_map_ledger_all_ok detail=1 when charge_ledger_ok on every point | op >= value 1.0 detail ledger ok on all points |
| charge-ledger-closes _(primary)_ | kind=derived path=charge_ledger_residual_max detail=worst-case charge residual | op <= value 1e-09 detail charge residual within tolerance |
| faradaic-efficiency-bounded _(primary)_ | kind=derived path=faradaic_efficiency_max detail=worst-case faradaic efficiency over the run | op <= value 1.000001 detail FE <= 1 (1e-6 rounding allowance) |
| current-voltage-residual _(diagnostic)_ | kind=derived path=iv_residual_max detail=worst-case |residual| / emf_fresh | op <= value 1e-06 detail circuit residual below 1e-6 |
| current-in-altair-window _(primary)_ | kind=derived path=settled_current_density detail=settled cell current density | op range low 1e-05 high 0.01 detail j in [1e-5, 1e-2] A/m^2 |
| selectivity-to-carbon _(diagnostic)_ | kind=derived path=settled_faradaic_efficiency detail=settled fraction of the current reducing CO2 | op >= value 0.5 detail faradaic efficiency to carbon >= 0.5 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Altair-2025 macroscale reactor currents ~10 nA .. 10 uA sufficient for carbon reduction | settled Butler-Volmer current density on a 1e-3 m^2 interface | Altair et al. 2025 (campaign dossier ref 12; parameter-registry electrochemical_current) | Used as the calibration window of the exchange currents (vg-data electrochemical_interface.observations.cell_current_density); 79.7% of the grid inside it. |
| Preiner 2020 product hierarchy formate >> acetate >> pyruvate, awaruite (Ni3Fe) the most productive catalyst | electron branch fractions and the Ni gain of the CO2-reduction exchange current | Preiner et al. 2020 | Branch fractions inferred from the yield hierarchy; the Ni gain is a hypothetical shape fitted to the ordering. |


## Study: The Open Network: Do Autocatalytic Motifs Emerge From Rule-Generated Chemistry, Beyond Chance? (`og-02-alt-reaction-universe`)

**Question.** Model Upgrade plan, Phase 3. og-02 showed a hand-curated rTCA fragment persists and self-amplifies under flow — but its autocatalytic turn (OAA + 2 CO2 -> 2 OAA + acetate) was written into the network. If the network is instead GENERATED from a Wolos-2020-style rule set over C1-C3 feedstock with real eQuilibrator thermodynamics, catalyst limits and degradation, and fed the Phase-1 posterior vent state, do persistent autocatalytic subnetworks appear at all, are they more frequent than in matched null networks, and how does the curated rTCA class compare when all three are graded by one post-hoc detector and one element / free-energy ledger?

**Objective.** Build the proto_metabolism face (curated_rtca | reaction_universe | null_network), the vg-data package reaction_universe (rules, eQuilibrator dG', catalyst requirements), the NetworkLedgerProcess (exact element residual, integrated carbon balance, dissipation >= 0) and a class-agnostic post-hoc motif detector; run the three classes under two feeds (Phase-1 posterior; + 10 mM formaldehyde) with 24 matched nulls per arm, an ignition map, flux-control analysis and a stiffness check.

**Hypothesis.** Under the Phase-1 posterior feed (CO2 1.4 mM, H2 10.8 mM, pyruvate 0.1 mM) the generated universe sustains a connected multi-species network but no autocatalytic motif (the C1 -> C2 bottleneck is thermodynamic at vent concentrations); supplying the formose feedstock (formaldehyde) lets a motif emerge from the rules and that motif count is enriched relative to degree-preserving, carbon/oxygen-matched nulls; the curated class's motif is detected but imposed and its stoichiometry fails the element ledger.

**Claim.** With real thermodynamics and conservation, autocatalysis is not free: the generated universe fed the Phase-1 vent posterior builds a persistent, element-balanced, dissipative network but no autocatalytic motif, because every C-C-forming route is uphill at vent concentrations; the one motif that does emerge — the formose cycle, recovered post hoc from the rules, absent in every matched null — needs a formaldehyde supply the vent chemistry does not make and a residence time shorter than its reductive destruction. The curated rTCA result survives as one class among three, with its motif now classed as imposed and its stoichiometry flagged as non-conserving.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- E: protometabolic network formation (rule-generated, not curated)
- D: surface chemistry as catalyst classes with finite site pools

**Proposed models / methods**

- Rule-based network generation (Wolos 2020 style) over linear C/H/O skeletons
- eQuilibrator component-contribution dG' + group-additivity fallback; detailed-balance kinetics
- libRoadRunner CVODE integration; post-hoc autocatalytic-core enumeration (Blokhuis 2020)
- Degree-preserving (C,O)-matched null networks (Maslov-Sneppen); empirical enrichment
- Element + free-energy ledger; flux-control summation theorem; tolerance / stiffness check

**Key assumptions**

- Linear C/H/O skeletons up to C4; eleven reaction rules (H2 reductions, enol / reductive carboxylation, formose initiation, aldol, isomerisation, alpha/beta decarboxylation, Cannizzaro); every reaction reversible with k_down = k0 (rule class) and k_up = k0 exp(-|dG'0|/RT).
- dG' potentials at pH 9.38 / I 0.25 M / 322 K (the Phase-1 posterior example point); water activity 1; CO2 and H2 vent-buffered, organics as CSTR inflow.
- Catalyst classes (mineral: reductions / carboxylations; base: aldol / isomerisation / Cannizzaro) share a finite site pool f = sites / (1 + load / K_sat), updated per tick.
- Null = degree-preserving swaps of organic species within (C, O)-count classes; dG' recomputed from the same potentials; same exchange / degradation reactions.
- A persistent autocatalytic motif = a simple cycle of flux-carrying reactions (|v| > 1e-12 M/s, species > 1e-9 M) with stoichiometric gain > 1 (Blokhuis 2020 type-I core); currency species CO2 / H2 / H2O excluded as cycle members.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Open chemical network — reaction_universe at the Phase-1 feed` | `viva_genesis.composites.open_chemical_network.open_chemical_network` | 0 | model_proto_metabolism=reaction_universe |

Declared parameter sets (`study.yaml` variants):

- **formose-feed** — `feed_formaldehyde=0.01`
- **null-network** — `model_proto_metabolism=null_network`, `null_seed=0`
- **curated-rtca** — `model_proto_metabolism=curated_rtca`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.open_chemical_network.open_chemical_network`**

og-02's continuous-feed reactor rebuilt as the Phase-3 open chemical network: the reaction network is a model CLASS on the proto_metabolism face (model_proto_metabolism: curated_rtca | reaction_universe | null_network), all three integrated by the genuine libRoadRunner engine under the same Phase-1-posterior feed (CO2, H2, pyruvate), graded by one class-agnostic post-hoc detector (flux-carrying simple cycles with stoichiometric gain > 1) and audited by a NetworkLedgerProcess (exact element residual, integrated carbon balance, dissipation >= 0). The universe class is generated from Wolos-2020-style rules over linear C/H/O skeletons with eQuilibrator dG' (vg-data package reaction_universe); the null class is its degree-preserving, carbon-matched rewiring.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_open_chemical_network_open_chemical_network = {
    'model_proto_metabolism': 'reaction_universe',   # Model class behind the 'proto_metabolism' face: curated_rtca | null_network | reaction_universe
    'feed_CO2'              : 0.0014,   # Vent-buffered CO2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_H2'               : 0.0108,   # Vent-buffered H2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_pyruvate'         : 0.0001,   # Pyruvate CSTR inflow (M) from og-01 reduced carbon [vg-data: inferred]
    'feed_formaldehyde'     : 0.0,   # Formaldehyde CSTR inflow (M); 0 canonical, 1e-3 in the formose arm [vg-data: hypothetical]
    'k_wash'                : 0.05,   # CSTR washout / dilution rate (1/s) [vg-data: inferred]
    'null_seed'             : 0,   # Null-network draw (null_network class only)
    'max_carbons'           : 4,   # Carbon cap of the generated universe (reaction_universe / null_network) [vg-data: hypothetical]
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `feed_CO2` | float | molar | >= 0.0 |
| `feed_H2` | float | molar | >= 0.0 |
| `feed_pyruvate` | float | molar | >= 0.0 |
| `feed_formaldehyde` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |
| `autocatalytic_gain` | float | dimensionless | >= 0.0 |
| `network_connected` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `element_balance` (invariant): `ledger.element_residual_max == 0`
- `carbon_balance` (invariant): `ledger.carbon_residual_max < 1e-4`
- `energy_directionality` (invariant): `ledger.free_energy_directionality == 1`
- `motifs_post_hoc` (invariant): `no rule names a cycle; motifs are detected by viva_genesis.protometabolism.motifs on every class`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-02-alt-reaction-universe ===
STUDY = 'og-02-alt-reaction-universe'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Open_chemical_network_reaction_universe_at_the_Phase_1_feed = 100
INTERVAL_Open_chemical_network_reaction_universe_at_the_Phase_1_feed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_open_chemical_network_open_chemical_network if 'ov_viva_genesis_composites_open_chemical_network_open_chemical_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_open_chemical_network_open_chemical_network = materialize_composite('viva_genesis.composites.open_chemical_network.open_chemical_network', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_open_chemical_network_open_chemical_network['state']}, core=core)
            _comp.run(STEPS_Open_chemical_network_reaction_universe_at_the_Phase_1_feed)
        print('ran', 'Open chemical network — reaction_universe at the Phase-1 feed', f'(viva_genesis.composites.open_chemical_network.open_chemical_network) for', STEPS_Open_chemical_network_reaction_universe_at_the_Phase_1_feed, 'steps')
    except Exception as _e:
        print(f"Open chemical network — reaction_universe at the Phase-1 feed not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**reaction-universe-enrichment**


In [ ]:
# reaction-universe-enrichment
show_viz(_render_one('local:ReactionUniverseEnrichment', {'title': 'Phase 3 — motif enrichment vs matched nulls, and where motifs ignite (Decision view)', 'report_json': 'workspace/studies/og-02-alt-reaction-universe/artifacts/motif_enrichment.json'}, RUNS_DB, STUDY_YAML))

**reaction-universe-subnetwork**


In [ ]:
# reaction-universe-subnetwork
show_viz(_render_one('local:ReactionUniverseSubnetwork', {'title': 'The persistent subnetwork and its post-hoc formose cycle (Mechanism view)', 'report_json': 'workspace/studies/og-02-alt-reaction-universe/artifacts/motif_enrichment.json', 'arm': 'formose_feed'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| network-ledger-closes _(primary)_ | kind=derived path=network_ledger_ok_final detail=ledger_ok on the final tick | op >= value 1.0 detail every ledger witness within tolerance |
| element-residual-exact _(diagnostic)_ | kind=derived path=max_element_residual detail=worst-case stoichiometric element residual | op <= value 1e-12 detail element balance exact by construction |
| free-energy-directionality _(diagnostic)_ | kind=derived path=free_energy_directionality_final detail=1 while dissipation has never been below -1e-12 W | op >= value 1.0 detail directionality witness holds through the run |
| persistent-network-at-phase1-feed _(primary)_ | kind=derived path=final_n_persistent_species detail=organic species above conc_floor on the final tick | op >= value 10.0 detail >= 10 persistent species |
| rtca-branch-emerges-from-rules _(supporting)_ | kind=derived path=settled_malate_M detail=settled malate (last quartile) | op > value 1e-09 detail malate persists above the species floor |
| no-motif-at-phase1-feed _(diagnostic)_ | kind=derived path=universe_motif_count detail=universe motif count, arm phase1_feed, from artifacts/motif_enrichment.json | op >= value 0.0 detail informative readout (any value passes) |
| motifs-emerge-not-imposed _(primary)_ | kind=derived path=motif_enrichment_p detail=empirical p of the universe motif count vs 24 nulls, arm formose_feed | op <= value 0.05 detail enrichment p <= 0.05 (with >= 1 motif see motif-present-formose-arm) None |
| motif-present-formose-arm _(primary)_ | kind=derived path=universe_motif_count detail=universe motif count, arm formose_feed | op >= value 1.0 detail >= 1 persistent motif |
| curated-rtca-fails-element-ledger _(diagnostic)_ | kind=derived path=curated_element_residual detail=max|E S| over the curated SBML (post hoc) | op > value 0.0 detail residual > 0 documents the violation (expected 8) |
| flux-control-summation-theorem _(diagnostic)_ | kind=derived path=fcc_summation_residual_max detail=from artifacts/reaction_universe_verification.json | op <= value 0.01 detail summation residual <= 1e-2 on dominant fluxes |
| integrator-tolerance-insensitive _(diagnostic)_ | kind=derived path=tolerance_sensitivity_max detail=worst relative species deviation across tolerance settings | op <= value 0.01 detail <= 1% relative deviation |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| The formose reaction is autocatalytic in glycolaldehyde via aldol / isomerisation / retro-aldol of the tetrose, with a slow umpolung initiation from formaldehyde alone | the detected motif's species and reactions | Breslow 1959 | Recovered post hoc from the rule set (glycolaldehyde -> glyceraldehyde -> DHA -> erythrulose -> erythrose -> 2 glycolaldehyde, gain 2) without being named by any rule. |
| Rule-generated prebiotic networks contain self-regenerating cycles that appear only after several synthetic generations | motif count vs matched nulls | Wolos 2020 | One cycle at C <= 4 in C/H/O; enrichment p = 0.04 over 24 nulls. |


## Open decisions
- Rule-set scope for the Phase-3 reaction universe: keep C1-C3 feedstock / C/H/O-only chemistry (as built: products capped at C4, 413 reactions) or include N/P-bearing reactions from the start, folding the og-res-N / og-res-P resource chemistries into the universe?
- Publication milestone: close the first paper at the pre-genetic arc (og-00 -> og-03) or extend through the early-genetic arc (og-04 protocells -> og-06 heredity)?
- First limiting resource / next module: which enrichment-plan module to build first — nitrogen assimilation (N2/NO3->NH4, FeS), phosphorus availability (phosphate/polyphosphate co-concentration), proto-energetics (thioester/acyl-phosphate energy currency before pumping), or metabolism->growth/division (osmotic growth)?
- Actually RUN the og-03 -> og-02 localization->ignition coupling (trap enrichment feeds the rTCA network as a real swept phase map), resolving the demoted phantom finding and giving og-03 its own Decision phase map?
- Reframe the campaign's headline thesis from 'cells could originate at alkaline vents' to the enrichment plan's 'quantitative MAP OF THE BOTTLENECKS between geology and cells' (dependencies, failure regimes, discriminating experiments)?
- Adopt the Full Modeling Plan architecture (emergence-not-presupposition + structural rewrites + canonical schemas + model substitution) as the campaign's build target, in the plan's section-18 order — canonical schemas + geochemical_world first, then the electrochemical interface, then C-N-P/energetics as cross-cutting layers, then localization, boundary, growth/division rewrites, competing heredity, ecology, autonomy?
- Adopt the Model Upgrade plan's three-quantity standard (feasibility / robustness / discrimination) and evidence tiers as the reporting requirement for every stage — i.e. a stage's verdict may not advance until it carries a TransitionReport whose decision-discipline audit is clean — and re-label the fourteen pre-upgrade studies (every member study except the already- tier-labelled og-cal) with their tier language?
- Attach the MARHYS 4.0 hydrothermal-fluid compilation (PANGAEA) as a workspace dataset so the Phase-1 conditioning set becomes the ultramafic-vent population rather than Lost City alone — and which Hadean-ocean prior (DIC, pH, T) should replace the modern-seawater proxy the audit flags?
